<a href="https://colab.research.google.com/github/Misha-private/Demo-repo/blob/main/1_layer_shallow_water_model_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted(): drive.mount('/content/drive')


Mounted at /content/drive


# build_1layer_ic_ctatalog_klein.py

In [ ]:
# ============================================================
# build_1layer_ic_ctatalog_klein.py
# ------------------------------------------------------------
# Build a 1-layer IC catalog for shallow-water experiments on
# a twisted/Klein-beta-plane C-grid.
#
# Output:
#   - bundle_cgrid_1L.npz   with h (centers), u (x-faces), v (y-faces)
#   - preview PNGs
#   - catalog_summary.txt
#
# Designed for Google Colab first; later reusable on HPC.
# ============================================================

import os
import math
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# User settings
# -------------------------
OUT_DIR = "/content/drive/MyDrive/AI_4DVAR/klein_ics_1L"
BUNDLE_NAME = "bundle_cgrid_1L.npz"


# Grid / domain
nx = 256
ny = 128
Lx = 2.0e7
Ly = 8.0e6
dx = Lx / nx
dy = Ly / ny

# 1-layer shallow-water parameters
g = 9.81
H = 1000.0
fp = 8.0e-5

# Catalog sizes
N_GAUSS = 6
N_VORT  = 6
N_WAVE  = 6
N_MIX   = 6

# Held-out test cases
N_RH_TEST = 2
N_MODON_TEST = 2

# Amplitude scales
ETA_AMP_TRAIN = 60.0     # meters
ETA_AMP_TEST  = 80.0     # meters
VEL_CAP       = 80.0     # m/s safety cap for generated IC winds

SEED = 0
rng = np.random.default_rng(SEED)

# -------------------------
# Geometry / grids
# -------------------------
x_c = np.linspace(0.5 * dx, Lx - 0.5 * dx, nx)
y_c = np.linspace(0.5 * dy, Ly - 0.5 * dy, ny)
Xc, Yc = np.meshgrid(x_c, y_c)

x_u = np.linspace(0.0, Lx, nx + 1)
y_v = np.linspace(0.0, Ly, ny + 1)
Xu, Yu = np.meshgrid(x_u, y_c)
Xv, Yv = np.meshgrid(x_c, y_v)

phi_c = np.pi * ((Yc / Ly) - 0.5)
f_c = fp * np.sin(phi_c)

phi_u = np.pi * ((Yu / Ly) - 0.5)
phi_v = np.pi * ((Yv / Ly) - 0.5)
f_u = fp * np.sin(phi_u)
f_v = fp * np.sin(phi_v)

# -------------------------
# Helpers
# -------------------------
def center_from_u(u):
    return 0.5 * (u[:, :-1] + u[:, 1:])

def center_from_v(v):
    return 0.5 * (v[:-1, :] + v[1:, :])

def uface_from_uc(uc):
    """Periodic in x."""
    u = np.zeros((ny, nx + 1), dtype=np.float32)
    u[:, 1:nx] = 0.5 * (uc[:, :-1] + uc[:, 1:])
    u[:, 0] = 0.5 * (uc[:, -1] + uc[:, 0])
    u[:, nx] = u[:, 0]
    return u

def vface_from_vc(vc):
    """Edge-like in y."""
    v = np.zeros((ny + 1, nx), dtype=np.float32)
    v[1:ny, :] = 0.5 * (vc[:-1, :] + vc[1:, :])
    v[0, :] = vc[0, :]
    v[ny, :] = vc[-1, :]
    return v

def twist_reflect_x(arr):
    return arr[..., ::-1]

def apply_bc_1l(u, v, h):
    # center h even
    h[0, :]  = 0.5 * (h[1, :]  + twist_reflect_x(h[1, :]))
    h[-1, :] = 0.5 * (h[-2, :] + twist_reflect_x(h[-2, :]))
    # u odd
    u[0, :]  = 0.5 * (u[1, :]  - twist_reflect_x(u[1, :]))
    u[-1, :] = 0.5 * (u[-2, :] - twist_reflect_x(u[-2, :]))
    # v even
    v[0, :]  = 0.5 * (v[1, :]  + twist_reflect_x(v[1, :]))
    v[-1, :] = 0.5 * (v[-2, :] + twist_reflect_x(v[-2, :]))
    return u, v, h

def ddx_center(a):
    return (np.roll(a, -1, axis=1) - np.roll(a, 1, axis=1)) / (2.0 * dx)

def ddy_center(a):
    a_up = np.vstack([a[0:1, :], a[:-1, :]])
    a_dn = np.vstack([a[1:, :], a[-1:, :]])
    return (a_dn - a_up) / (2.0 * dy)

def geostrophic_uv_from_eta(eta, fmin=2.0e-5):
    """
    Simple geostrophic-ish balanced center velocities:
       u = -(g/f) d eta / d y
       v =  (g/f) d eta / d x
    Use clipped |f| to avoid equatorial blow-up.
    """
    f_eff = np.sign(f_c) * np.maximum(np.abs(f_c), fmin)
    deta_dx = ddx_center(eta)
    deta_dy = ddy_center(eta)

    uc = -(g / f_eff) * deta_dy
    vc =  (g / f_eff) * deta_dx

    uc = np.clip(uc, -VEL_CAP, VEL_CAP)
    vc = np.clip(vc, -VEL_CAP, VEL_CAP)
    return uc.astype(np.float32), vc.astype(np.float32)

def gaussian2d(x0, y0, sx, sy):
    rx = (Xc - x0) / sx
    ry = (Yc - y0) / sy
    return np.exp(-(rx**2 + ry**2)).astype(np.float32)

def cosine_wave(kx, ky, phase_x=0.0, phase_y=0.0):
    return np.cos(2*np.pi*kx*Xc/Lx + phase_x) * np.cos(2*np.pi*ky*Yc/Ly + phase_y)

def save_preview(name, h, u, v, out_dir):
    uc = center_from_u(u)
    vc = center_from_v(v)
    eta = h - H

    fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))
    im = axs[0].imshow(eta, origin="lower")
    axs[0].set_title(f"{name}: eta")
    plt.colorbar(im, ax=axs[0], fraction=0.046, pad=0.04)

    im = axs[1].imshow(uc, origin="lower")
    axs[1].set_title("u center")
    plt.colorbar(im, ax=axs[1], fraction=0.046, pad=0.04)

    im = axs[2].imshow(vc, origin="lower")
    axs[2].set_title("v center")
    plt.colorbar(im, ax=axs[2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{name}.png"), dpi=140)
    plt.close()

# -------------------------
# IC families
# -------------------------
def make_gaussian_ic(i):
    amp = rng.uniform(-ETA_AMP_TRAIN, ETA_AMP_TRAIN)
    x0 = rng.uniform(0.15*Lx, 0.85*Lx)
    y0 = rng.uniform(0.20*Ly, 0.80*Ly)
    sx = rng.uniform(0.05*Lx, 0.15*Lx)
    sy = rng.uniform(0.05*Ly, 0.18*Ly)

    eta = amp * gaussian2d(x0, y0, sx, sy)
    uc, vc = geostrophic_uv_from_eta(eta)
    h = (H + eta).astype(np.float32)
    u = uface_from_uc(uc)
    v = vface_from_vc(vc)
    return f"gauss_{i:02d}", apply_bc_1l(u, v, h)

def make_vortex_ic(i):
    amp = rng.uniform(0.5*ETA_AMP_TRAIN, ETA_AMP_TRAIN)
    x0 = rng.uniform(0.2*Lx, 0.8*Lx)
    y0 = rng.uniform(0.2*Ly, 0.8*Ly)
    sx = rng.uniform(0.06*Lx, 0.12*Lx)
    sy = rng.uniform(0.06*Ly, 0.12*Ly)

    core = gaussian2d(x0, y0, sx, sy)
    ring = gaussian2d(x0, y0, 1.8*sx, 1.8*sy)
    eta = amp * (core - 0.7*ring)
    uc, vc = geostrophic_uv_from_eta(eta)
    h = (H + eta).astype(np.float32)
    u = uface_from_uc(uc)
    v = vface_from_vc(vc)
    return f"vort_{i:02d}", apply_bc_1l(u, v, h)

def make_wave_ic(i):
    amp = rng.uniform(0.5*ETA_AMP_TRAIN, ETA_AMP_TRAIN)
    kx = rng.integers(1, 5)
    ky = rng.integers(1, 4)
    phx = rng.uniform(0, 2*np.pi)
    phy = rng.uniform(0, 2*np.pi)

    eta = amp * cosine_wave(kx, ky, phx, phy).astype(np.float32)
    uc, vc = geostrophic_uv_from_eta(eta)
    h = (H + eta).astype(np.float32)
    u = uface_from_uc(uc)
    v = vface_from_vc(vc)
    return f"wave_{i:02d}", apply_bc_1l(u, v, h)

def make_mixed_ic(i):
    amp1 = rng.uniform(0.4*ETA_AMP_TRAIN, ETA_AMP_TRAIN)
    amp2 = rng.uniform(-0.5*ETA_AMP_TRAIN, 0.5*ETA_AMP_TRAIN)

    x0 = rng.uniform(0.2*Lx, 0.8*Lx)
    y0 = rng.uniform(0.2*Ly, 0.8*Ly)
    sx = rng.uniform(0.06*Lx, 0.14*Lx)
    sy = rng.uniform(0.06*Ly, 0.14*Ly)

    bump = gaussian2d(x0, y0, sx, sy)
    wave = cosine_wave(rng.integers(1,4), rng.integers(1,3),
                       rng.uniform(0,2*np.pi), rng.uniform(0,2*np.pi))
    eta = (amp1*bump + amp2*wave).astype(np.float32)

    uc, vc = geostrophic_uv_from_eta(eta)
    h = (H + eta).astype(np.float32)
    u = uface_from_uc(uc)
    v = vface_from_vc(vc)
    return f"mix_{i:02d}", apply_bc_1l(u, v, h)

# -------------------------
# Held-out test families
# -------------------------
def make_rh_test_ic(i):
    """
    A Rossby-Haurwitz-like large-scale pattern for held-out testing.
    This is not a full spherical RH exact solution, but a large-scale
    wave-like analog suitable for planar/twisted-beta testing.
    """
    m = int(rng.integers(3, 6))
    amp = rng.uniform(0.7*ETA_AMP_TEST, ETA_AMP_TEST)

    base = np.cos(2*np.pi*m*Xc/Lx) * np.cos(np.pi*(Yc/Ly - 0.5))
    shear = 0.35 * np.sin(4*np.pi*Xc/Lx) * np.cos(2*np.pi*Yc/Ly)
    eta = amp * (base + shear).astype(np.float32)

    uc, vc = geostrophic_uv_from_eta(eta)
    h = (H + eta).astype(np.float32)
    u = uface_from_uc(uc)
    v = vface_from_vc(vc)
    return f"test_rh_{i:02d}", apply_bc_1l(u, v, h)

def make_modon_test_ic(i):
    """
    Dipole/modon-like held-out test.
    """
    amp = rng.uniform(0.7*ETA_AMP_TEST, ETA_AMP_TEST)
    x0 = rng.uniform(0.30*Lx, 0.70*Lx)
    y0 = rng.uniform(0.30*Ly, 0.70*Ly)
    sx = rng.uniform(0.05*Lx, 0.10*Lx)
    sy = rng.uniform(0.05*Ly, 0.10*Ly)

    left  = gaussian2d(x0 - 0.8*sx, y0, sx, sy)
    right = gaussian2d(x0 + 0.8*sx, y0, sx, sy)
    eta = amp * (left - right).astype(np.float32)

    uc, vc = geostrophic_uv_from_eta(eta)
    h = (H + eta).astype(np.float32)
    u = uface_from_uc(uc)
    v = vface_from_vc(vc)
    return f"test_modon_{i:02d}", apply_bc_1l(u, v, h)

# -------------------------
# Build catalog
# -------------------------
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    preview_dir = os.path.join(OUT_DIR, "previews")
    os.makedirs(preview_dir, exist_ok=True)

    bundle = {}
    summary_lines = []

    # training families
    makers = (
        [(make_gaussian_ic, N_GAUSS)] +
        [(make_vortex_ic,   N_VORT)]  +
        [(make_wave_ic,     N_WAVE)]  +
        [(make_mixed_ic,    N_MIX)]
    )

    for maker, n in makers:
        for i in range(n):
            key, (u, v, h) = maker(i)
            h = np.maximum(h, 5.0).astype(np.float32)
            bundle[f"{key}_h"] = h
            bundle[f"{key}_u"] = u.astype(np.float32)
            bundle[f"{key}_v"] = v.astype(np.float32)

            eta = h - H
            summary_lines.append(
                f"{key:16s} TRAIN  "
                f"eta[min,max,std]=({eta.min():8.2f},{eta.max():8.2f},{eta.std():8.2f})  "
                f"u_std={center_from_u(u).std():8.2f}  v_std={center_from_v(v).std():8.2f}"
            )
            save_preview(key, h, u, v, preview_dir)

    # held-out test families
    test_makers = (
        [(make_rh_test_ic,    N_RH_TEST)] +
        [(make_modon_test_ic, N_MODON_TEST)]
    )

    for maker, n in test_makers:
        for i in range(n):
            key, (u, v, h) = maker(i)
            h = np.maximum(h, 5.0).astype(np.float32)
            bundle[f"{key}_h"] = h
            bundle[f"{key}_u"] = u.astype(np.float32)
            bundle[f"{key}_v"] = v.astype(np.float32)

            eta = h - H
            summary_lines.append(
                f"{key:16s} TEST   "
                f"eta[min,max,std]=({eta.min():8.2f},{eta.max():8.2f},{eta.std():8.2f})  "
                f"u_std={center_from_u(u).std():8.2f}  v_std={center_from_v(v).std():8.2f}"
            )
            save_preview(key, h, u, v, preview_dir)

    # metadata
    bundle["nx"] = np.int32(nx)
    bundle["ny"] = np.int32(ny)
    bundle["dx"] = np.float32(dx)
    bundle["dy"] = np.float32(dy)
    bundle["Lx"] = np.float32(Lx)
    bundle["Ly"] = np.float32(Ly)
    bundle["g"] = np.float32(g)
    bundle["H"] = np.float32(H)
    bundle["fp"] = np.float32(fp)
    bundle["x_c"] = x_c.astype(np.float32)
    bundle["y_c"] = y_c.astype(np.float32)
    bundle["f_c"] = f_c.astype(np.float32)

    bundle_path = os.path.join(OUT_DIR, BUNDLE_NAME)
    np.savez_compressed(bundle_path, **bundle)

    summary_path = os.path.join(OUT_DIR, "catalog_summary.txt")
    with open(summary_path, "w") as f:
        f.write("\n".join(summary_lines))

    print("Saved bundle:", bundle_path)
    print("Saved summary:", summary_path)
    print("Saved previews in:", preview_dir)
    print("Number of ICs:", len([k for k in bundle if k.endswith("_h")]))

if __name__ == "__main__":
    main()


Saved bundle: /content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz
Saved summary: /content/drive/MyDrive/AI_4DVAR/klein_ics_1L/catalog_summary.txt
Saved previews in: /content/drive/MyDrive/AI_4DVAR/klein_ics_1L/previews
Number of ICs: 28


# Run all ICs with FD model

In [ ]:
# ============================================================
# run_fd_1layer_catalog.py
# ------------------------------------------------------------
# Run 1-layer shallow-water FD model on twisted/Klein beta-plane
# for every IC in a C-grid bundle.
#
# INPUT:
#   /content/drive/MyDrive/klein_ics_1L/bundle_cgrid_1L.npz
#
# OUTPUT:
#   /content/drive/MyDrive/klein_ckpt_1L/<IC_KEY>/klein_step_XXXXXX.npz
#   plus plots and time series
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Paths
# -------------------------
IC_BUNDLE = "/content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz"
ROOT_OUT  = "/content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L"

# -------------------------
# Physics / grid
# -------------------------
g = 9.81
H = 1000.0
nx = 256
ny = 128
Lx = 2.0e7
Ly = 8.0e6
dx = Lx / nx
dy = Ly / ny
fp = 8.0e-5

dt = 30.0
nt = 1200
SAVE_STEPS = {0, 200, 400, 600, 800, 1000, 1200}

# diffusion
nu2_u, nu2_v, nu2_h = 1.0e4, 1.0e4, 5.0e3
nu4_u, nu4_v, nu4_h = 5.0e10, 5.0e10, 2.5e10

HMIN = 0.5

# -------------------------
# Geometry
# -------------------------
x_c = np.linspace(0.5*dx, Lx-0.5*dx, nx)
y_c = np.linspace(0.5*dy, Ly-0.5*dy, ny)
Xc, Yc = np.meshgrid(x_c, y_c)
phi_c = np.pi*((Yc/Ly) - 0.5)
f_c = fp*np.sin(phi_c)

x_u = np.linspace(0.0, Lx, nx+1)
y_v = np.linspace(0.0, Ly, ny+1)
Xu, Yu = np.meshgrid(x_u, y_c)
Xv, Yv = np.meshgrid(x_c, y_v)
phi_u = np.pi*((Yu/Ly) - 0.5)
phi_v = np.pi*((Yv/Ly) - 0.5)
f_u = fp*np.sin(phi_u)
f_v = fp*np.sin(phi_v)

# -------------------------
# Klein twist BCs
# -------------------------
def twist_reflect_x(arr):
    return arr[..., ::-1]

def apply_bc_1l(u, v, h):
    # centers even
    h[0, :]  = 0.5*(h[1, :]  + twist_reflect_x(h[1, :]))
    h[-1, :] = 0.5*(h[-2, :] + twist_reflect_x(h[-2, :]))
    # u odd
    u[0, :]  = 0.5*(u[1, :]  - twist_reflect_x(u[1, :]))
    u[-1, :] = 0.5*(u[-2, :] - twist_reflect_x(u[-2, :]))
    # v even
    v[0, :]  = 0.5*(v[1, :]  + twist_reflect_x(v[1, :]))
    v[-1, :] = 0.5*(v[-2, :] + twist_reflect_x(v[-2, :]))
    return u, v, h

# -------------------------
# C-grid helpers
# -------------------------
def center_from_u(u):
    return 0.5*(u[:, :-1] + u[:, 1:])

def center_from_v(v):
    return 0.5*(v[:-1, :] + v[1:, :])

def avg_x(a):
    return 0.5*(np.pad(a, ((0,0),(1,0)), mode='wrap') + np.pad(a, ((0,0),(0,1)), mode='wrap'))

def avg_y(a):
    return 0.5*(np.pad(a, ((1,0),(0,0)), mode='edge') + np.pad(a, ((0,1),(0,0)), mode='edge'))

def ddx_c_to_u(phi):
    L = np.pad(phi, ((0,0),(1,0)), mode='wrap')
    R = np.pad(phi, ((0,0),(0,1)), mode='wrap')
    return (R - L) / (2.0*dx)

def ddy_c_to_v(phi):
    T = np.pad(phi, ((1,0),(0,0)), mode='edge')
    B = np.pad(phi, ((0,1),(0,0)), mode='edge')
    return (B - T) / (2.0*dy)

def ddx_u_to_c(phi_u):
    return (phi_u[:, 1:] - phi_u[:, :-1]) / dx

def ddy_v_to_c(phi_v):
    return (phi_v[1:, :] - phi_v[:-1, :]) / dy

# -------------------------
# Laplacians
# -------------------------
def lap_u(u):
    ue = np.pad(u, ((0,0),(1,1)), mode='wrap')
    u_xx = (ue[:, :-2] - 2*ue[:, 1:-1] + ue[:, 2:]) / dx**2
    ue2 = np.pad(u, ((1,1),(0,0)), mode='edge')
    u_yy = (ue2[:-2, :] - 2*ue2[1:-1, :] + ue2[2:, :]) / dy**2
    return u_xx + u_yy

def lap_v(v):
    ve = np.pad(v, ((0,0),(1,1)), mode='wrap')
    v_xx = (ve[:, :-2] - 2*ve[:, 1:-1] + ve[:, 2:]) / dx**2
    ve2 = np.pad(v, ((1,1),(0,0)), mode='edge')
    v_yy = (ve2[:-2, :] - 2*ve2[1:-1, :] + ve2[2:, :]) / dy**2
    return v_xx + v_yy

def lap_c(h):
    he = np.pad(h, ((0,0),(1,1)), mode='wrap')
    h_xx = (he[:, :-2] - 2*he[:, 1:-1] + he[:, 2:]) / dx**2
    he2 = np.pad(h, ((1,1),(0,0)), mode='edge')
    h_yy = (he2[:-2, :] - 2*he2[1:-1, :] + he2[2:, :]) / dy**2
    return h_xx + h_yy

def bih_u(u): return lap_u(lap_u(u))
def bih_v(v): return lap_v(lap_v(v))
def bih_c(h): return lap_c(lap_c(h))

# -------------------------
# Vorticity
# -------------------------
def compute_corner_vort(u, v):
    v_w = np.pad(v, ((0,0),(1,0)), mode='wrap')
    v_e = np.pad(v, ((0,0),(0,1)), mode='wrap')
    dv_dx = (v_e - v_w) / (2*dx)

    u_s = np.pad(u, ((1,0),(0,0)), mode='edge')
    u_n = np.pad(u, ((0,1),(0,0)), mode='edge')
    du_dy = (u_n - u_s) / (2*dy)

    return dv_dx - du_dy

def to_u_from_corners(a):
    return 0.5*(a[:-1, :] + a[1:, :])

def to_v_from_corners(a):
    return 0.5*(a[:, :-1] + a[:, 1:])

# -------------------------
# Positivity floor
# -------------------------
def enforce_floor_ke_preserving(u, v, h, hmin=HMIN):
    mask = (h < hmin)
    if not np.any(mask):
        return u, v, h, 0

    s_c = np.ones_like(h, dtype=np.float32)
    s_c[mask] = np.sqrt(np.maximum(h[mask], 1e-12) / hmin)

    s_u = 0.5*(np.pad(s_c, ((0,0),(1,0)), mode='wrap') + np.pad(s_c, ((0,0),(0,1)), mode='wrap'))
    s_v = 0.5*(np.pad(s_c, ((1,0),(0,0)), mode='edge') + np.pad(s_c, ((0,1),(0,0)), mode='edge'))

    u = u * s_u
    v = v * s_v
    h = np.maximum(h, hmin)
    return u, v, h, int(mask.sum())

# -------------------------
# RHS (1-layer AL-style)
# -------------------------
def rhs_1l(u, v, h):
    u, v, h = apply_bc_1l(u, v, h)

    uc = center_from_u(u)
    vc = center_from_v(v)
    K = 0.5*(uc**2 + vc**2)

    eta = h - H
    Phi = g * eta

    dPhidx_u = ddx_c_to_u(Phi)
    dPhidy_v = ddy_c_to_v(Phi)
    dKdx_u   = ddx_c_to_u(K)
    dKdy_v   = ddy_c_to_v(K)

    z_corners = compute_corner_vort(u, v)
    eta_u = to_u_from_corners(z_corners) + f_u
    eta_v = to_v_from_corners(z_corners) + f_v

    v_tu = avg_x(center_from_v(v))
    u_tv = avg_y(center_from_u(u))

    du = -(dPhidx_u + dKdx_u) + eta_u * v_tu + nu2_u*lap_u(u) + nu4_u*bih_u(u)
    dv = -(dPhidy_v + dKdy_v) - eta_v * u_tv + nu2_v*lap_v(v) + nu4_v*bih_v(v)

    h_u = avg_x(h)
    h_v = avg_y(h)
    F_u = h_u * u
    F_v = h_v * v

    dhdt = -(ddx_u_to_c(F_u) + ddy_v_to_c(F_v)) + nu2_h*lap_c(h) + nu4_h*bih_c(h)

    return apply_bc_1l(du, dv, dhdt)

# -------------------------
# RK4
# -------------------------
def rk4_1l(u, v, h, dt):
    k1u, k1v, k1h = rhs_1l(u, v, h)

    ub, vb, hb = apply_bc_1l(u + 0.5*dt*k1u, v + 0.5*dt*k1v, h + 0.5*dt*k1h)
    k2u, k2v, k2h = rhs_1l(ub, vb, hb)

    uc, vc, hc = apply_bc_1l(u + 0.5*dt*k2u, v + 0.5*dt*k2v, h + 0.5*dt*k2h)
    k3u, k3v, k3h = rhs_1l(uc, vc, hc)

    ud, vd, hd = apply_bc_1l(u + dt*k3u, v + dt*k3v, h + dt*k3h)
    k4u, k4v, k4h = rhs_1l(ud, vd, hd)

    u_new = u + (dt/6.0)*(k1u + 2*k2u + 2*k3u + k4u)
    v_new = v + (dt/6.0)*(k1v + 2*k2v + 2*k3v + k4h)
    h_new = h + (dt/6.0)*(k1h + 2*k2h + 2*k3h + k4h)

    return apply_bc_1l(u_new, v_new, h_new)

# -------------------------
# Diagnostics / save
# -------------------------
def total_mass(h):
    return float(np.sum(h) * dx * dy)

def total_energy(u, v, h):
    uc = center_from_u(u)
    vc = center_from_v(v)
    eta = h - H
    ke = 0.5*h*(uc**2 + vc**2)
    pe = 0.5*g*(eta**2)
    return float(np.sum((ke + pe) * dx * dy))

def save_centered_1L(ic_key, step, u, v, h, t):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    os.makedirs(ic_dir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    path = os.path.join(ic_dir, f"klein_step_{step:06d}.npz")
    np.savez_compressed(
        path,
        eta=eta.astype(np.float32),
        uc=uc.astype(np.float32),
        vc=vc.astype(np.float32),
        h=h.astype(np.float32),
        f=f_c.astype(np.float32),
        y_m=y_c.astype(np.float32),
        H=np.float32(H),
        dt=np.float32(dt),
        t=np.float32(t),
        nx=np.int32(nx), ny=np.int32(ny),
        dx=np.float32(dx), dy=np.float32(dy), fp=np.float32(fp),
    )
    return path

def quick_plot_1L(ic_key, step, u, v, h):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    xkm = Xc / 1e3
    ykm = (Yc - 0.5*Ly) / 1e3

    fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))
    im = axs[0].pcolormesh(xkm, ykm, eta, shading="auto")
    axs[0].set_title("eta (m)")
    plt.colorbar(im, ax=axs[0])

    im = axs[1].pcolormesh(xkm, ykm, uc, shading="auto")
    axs[1].set_title("u_c (m/s)")
    plt.colorbar(im, ax=axs[1])

    im = axs[2].pcolormesh(xkm, ykm, vc, shading="auto")
    axs[2].set_title("v_c (m/s)")
    plt.colorbar(im, ax=axs[2])

    plt.tight_layout()
    plt.savefig(os.path.join(pdir, f"fields_step_{step:06d}.png"), dpi=120)
    plt.close()

def plot_mass_energy(ic_key, steps, m_series, e_series):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    tdays = np.asarray(steps) * dt / 86400.0
    M0 = m_series[0]
    E0 = e_series[0]

    plt.figure(figsize=(8,4))
    plt.plot(tdays, (np.asarray(m_series)-M0)/M0, label="ΔM/M0")
    plt.plot(tdays, (np.asarray(e_series)-E0)/E0, label="ΔE/E0")
    plt.xlabel("time (days)")
    plt.ylabel("relative change")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "mass_energy_timeseries.png"), dpi=120)
    plt.close()

# -------------------------
# Load bundle and IC keys
# -------------------------
def list_ic_keys(bundle_path):
    d = np.load(bundle_path)
    keys = sorted(set(k[:-2] for k in d.files if k.endswith("_h")))
    return keys

def load_ic(bundle_path, ic_key):
    d = np.load(bundle_path)
    h = d[f"{ic_key}_h"].astype(np.float32)
    u = d[f"{ic_key}_u"].astype(np.float32)
    v = d[f"{ic_key}_v"].astype(np.float32)
    return h, u, v

# -------------------------
# Run one IC
# -------------------------
def run_ic(ic_key):
    print(f"\n=== 1L IC: {ic_key} | nx={nx}, ny={ny}, dt={dt:.1f}s, nt={nt} ===")
    h, u, v = load_ic(IC_BUNDLE, ic_key)
    u, v, h = apply_bc_1l(u, v, h)
    h[:] = np.maximum(h, 5.0)

    steps = []
    m_series = []
    e_series = []

    M0 = total_mass(h)
    E0 = total_energy(u, v, h)
    t = 0.0

    steps.append(0)
    m_series.append(M0)
    e_series.append(E0)

    if 0 in SAVE_STEPS:
        p = save_centered_1L(ic_key, 0, u, v, h, t)
        quick_plot_1L(ic_key, 0, u, v, h)
        print(f"[save] {ic_key} step=0 -> {p}")

    for n in range(1, nt+1):
        u, v, h = rk4_1l(u, v, h, dt)
        t += dt

        u, v, h, _ = enforce_floor_ke_preserving(u, v, h, HMIN)

        steps.append(n)
        m_series.append(total_mass(h))
        e_series.append(total_energy(u, v, h))

        if n in SAVE_STEPS:
            p = save_centered_1L(ic_key, n, u, v, h, t)
            quick_plot_1L(ic_key, n, u, v, h)
            print(f"[save] {ic_key} step={n} -> {p}")

        if (n % 100) == 0 or n == 1:
            uc = center_from_u(u)
            vc = center_from_v(v)
            umax = float(np.max(np.sqrt(uc*uc + vc*vc)))
            dM = (m_series[-1] - M0) / M0
            dE = (e_series[-1] - E0) / E0
            print(f"[{n:5d}] dM/M0={dM:+.3e}  dE/E0={dE:+.3e}  umax={umax:6.2f}")

    plot_mass_energy(ic_key, steps, m_series, e_series)
    print(f"Done (1L): {ic_key}")

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    os.makedirs(ROOT_OUT, exist_ok=True)
    keys = list_ic_keys(IC_BUNDLE)
    print("ICs in bundle:", keys)
    for k in keys:
        run_ic(k)


ICs in bundle: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']

=== 1L IC: gauss_00 | nx=256, ny=128, dt=30.0s, nt=1200 ===
[save] gauss_00 step=0 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000000.npz


ValueError: operands could not be broadcast together with shapes (129,256) (128,256) 

# build_ml_dataset_1layer.py

In [ ]:
# ============================================================
# build_ml_dataset_1layer.py
# ------------------------------------------------------------
# Build train/val/test ML datasets from 1-layer FD trajectories.
#
# INPUT:
#   /content/drive/MyDrive/klein_ckpt_1L/<IC_KEY>/klein_step_*.npz
#
# OUTPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ml_1L/
#       dataset_train_1L.npz
#       dataset_val_1L.npz
#       dataset_test_1L.npz
#       dataset_summary.txt
#
# Each sample:
#   X = [eta, uc, vc, f, y_norm] at time t
#   Y = [eta, uc, vc] at time t+ΔT
# ============================================================

import os
import glob
import numpy as np

# -------------------------
# Paths
# -------------------------
ROOT_IN = "/content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L"
ROOT_OUT = "/content/drive/MyDrive/AI_4DVAR/klein_ml_1L"
os.makedirs(ROOT_OUT, exist_ok=True)

# -------------------------
# Settings
# -------------------------
SAVE_STEPS = [0, 200, 400, 600, 800, 1000, 1200]
SEED = 0
VAL_FRACTION = 0.2   # fraction of non-test ICs used for validation

rng = np.random.default_rng(SEED)

# -------------------------
# Helpers
# -------------------------
def list_ic_dirs(root):
    # Debug print statement
    print(f"Checking for IC directories in: {root}")
    if not os.path.exists(root):
        print(f"Error: ROOT_IN directory does not exist: {root}")
        return []
    if not os.path.isdir(root):
        print(f"Error: ROOT_IN is not a directory: {root}")
        return []
    print(f"Contents of {root}: {os.listdir(root)}")

    dirs = sorted([d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)])
    return [os.path.basename(d) for d in dirs]

def load_step(ic_key, step):
    p = os.path.join(ROOT_IN, ic_key, f"klein_step_{step:06d}.npz")
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    d = np.load(p)
    return {k: d[k] for k in d.files}

def stack_input(d):
    """
    Input channels at time t:
      [eta, uc, vc, f, y_norm]
    Output shape: (5, ny, nx)
    """
    eta = d["eta"].astype(np.float32)
    uc  = d["uc"].astype(np.float32)
    vc  = d["vc"].astype(np.float32)
    f   = d["f"].astype(np.float32)

    y_m = d["y_m"].astype(np.float32)   # (ny,)
    ny, nx = eta.shape
    y2d = np.repeat(y_m[:, None], nx, axis=1)
    y_norm = (y2d - y2d.mean()) / (np.std(y2d) + 1e-8)
    y_norm = y_norm.astype(np.float32)

    X = np.stack([eta, uc, vc, f, y_norm], axis=0)
    return X

def stack_target(d):
    """
    Target channels at time t+ΔT:
      [eta, uc, vc]
    Output shape: (3, ny, nx)
    """
    eta = d["eta"].astype(np.float32)
    uc  = d["uc"].astype(np.float32)
    vc  = d["vc"].astype(np.float32)
    Y = np.stack([eta, uc, vc], axis=0)
    return Y

def classify_ic(ic_key):
    """
    Split logic:
      - test_rh_* and test_modon_* -> OOD test
      - all others -> pool for train/val
    """
    if ic_key.startswith("test_rh_") or ic_key.startswith("test_modon_"):
        return "test"
    return "pool"

# -------------------------
# Build IC split
# -------------------------
all_ics = list_ic_dirs(ROOT_IN)
pool_ics = []
test_ics = []

for key in all_ics:
    cls = classify_ic(key)
    if cls == "test":
        test_ics.append(key)
    else:
        pool_ics.append(key)

pool_ics = sorted(pool_ics)
test_ics = sorted(test_ics)

# validation split from pool
n_val = max(1, int(round(len(pool_ics) * VAL_FRACTION)))
perm = rng.permutation(len(pool_ics))
val_idx = set(perm[:n_val])

val_ics = [pool_ics[i] for i in range(len(pool_ics)) if i in val_idx]
train_ics = [pool_ics[i] for i in range(len(pool_ics)) if i not in val_idx]

train_ics = sorted(train_ics)
val_ics = sorted(val_ics)

print("Train ICs:", train_ics)
print("Val ICs  :", val_ics)
print("Test ICs :", test_ics)

# -------------------------
# Build sample lists
# -------------------------
def build_samples(ic_list):
    X_list, Y_list = [], []
    meta_ic = []
    meta_t0 = []
    meta_t1 = []

    for ic_key in ic_list:
        for s0, s1 in zip(SAVE_STEPS[:-1], SAVE_STEPS[1:]):
            d0 = load_step(ic_key, s0)
            d1 = load_step(ic_key, s1)

            X = stack_input(d0)
            Y = stack_target(d1)

            X_list.append(X)
            Y_list.append(Y)
            meta_ic.append(ic_key)
            meta_t0.append(s0)
            meta_t1.append(s1)

    # Only try to stack if lists are not empty
    if X_list and Y_list:
        X = np.stack(X_list, axis=0).astype(np.float32)   # (N, 5, ny, nx)
        Y = np.stack(Y_list, axis=0).astype(np.float32)   # (N, 3, ny, nx)
    else:
        # Return empty arrays with appropriate shapes if no data was collected
        # This prevents the ValueError from np.stack on empty lists
        # Assuming a default shape if no data is found, e.g., (0, 5, ny, nx) or handle it downstream.
        # For now, let's return empty arrays and let subsequent checks handle it if needed.
        # A more robust solution might involve knowing the expected ny, nx from global variables.
        # For this specific case, the `ValueError` will still occur if X_list is empty, but we've added a print for debugging the root cause.
        # Returning empty arrays will prevent the current np.stack ValueError, but the problem of no ICs still needs to be resolved.
        print("Warning: No samples generated for dataset.")
        return {"X": np.array([]), "Y": np.array([]), "ic_key": np.array([]), "step0": np.array([]), "step1": np.array([])}

    return {
        "X": X,
        "Y": Y,
        "ic_key": np.array(meta_ic),
        "step0": np.array(meta_t0, dtype=np.int32),
        "step1": np.array(meta_t1, dtype=np.int32),
    }

train = build_samples(train_ics)
val   = build_samples(val_ics)
test  = build_samples(test_ics)

# -------------------------
# Save datasets
# -------------------------
def save_dataset(path, data):
    # Only save if there's actual data to save
    if data["X"].size > 0:
        np.savez_compressed(path, **data)
    else:
        print(f"Warning: Not saving empty dataset to {path}")

train_path = os.path.join(ROOT_OUT, "dataset_train_1L.npz")
val_path   = os.path.join(ROOT_OUT, "dataset_val_1L.npz")
test_path  = os.path.join(ROOT_OUT, "dataset_test_1L.npz")

save_dataset(train_path, train)
save_dataset(val_path, val)
save_dataset(test_path, test)

# -------------------------
# Summary
# -------------------------
summary = []
summary.append(f"ROOT_IN  = {ROOT_IN}")
summary.append(f"ROOT_OUT = {ROOT_OUT}")
summary.append("")
summary.append(f"Train ICs ({len(train_ics)}): {train_ics}")
summary.append(f"Val ICs   ({len(val_ics)}): {val_ics}")
summary.append(f"Test ICs  ({len(test_ics)}): {test_ics}")
summary.append("")

# Check if datasets are empty before trying to access shape
if train["X"].size > 0: summary.append(f"Train X shape: {train['X'].shape}")
else: summary.append("Train X shape: (Empty)")
if train["Y"].size > 0: summary.append(f"Train Y shape: {train['Y'].shape}")
else: summary.append("Train Y shape: (Empty)")

if val["X"].size > 0: summary.append(f"Val   X shape: {val['X'].shape}")
else: summary.append("Val   X shape: (Empty)")
if val["Y"].size > 0: summary.append(f"Val   Y shape: {val['Y'].shape}")
else: summary.append("Val   Y shape: (Empty)")

if test["X"].size > 0: summary.append(f"Test  X shape: {test['X'].shape}")
else: summary.append("Test  X shape: (Empty)")
if test["Y"].size > 0: summary.append(f"Test  Y shape: {test['Y'].shape}")
else: summary.append("Test  Y shape: (Empty)")

summary.append("")
summary.append("Input channels : [eta, uc, vc, f, y_norm]")
summary.append("Target channels: [eta, uc, vc]")

summary_path = os.path.join(ROOT_OUT, "dataset_summary.txt")
with open(summary_path, "w") as f:
    f.write("\n".join(summary))

print("\nSaved:")
print(train_path)
print(val_path)
print(test_path)
print(summary_path)

print("\nShapes:")
if train["X"].size > 0: print("train X:", train["X"].shape, "train Y:", train["Y"].shape)
else: print("train X: (Empty) train Y: (Empty)")
if val["X"].size > 0: print("val   X:", val["X"].shape,   "val   Y:", val["Y"].shape)
else: print("val   X: (Empty) val   Y: (Empty)")
if test["X"].size > 0: print("test  X:", test["X"].shape,  "test  Y:", test["Y"].shape)
else: print("test  X: (Empty) test  Y: (Empty)")


Checking for IC directories in: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
Error: ROOT_IN directory does not exist: /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L
Train ICs: []
Val ICs  : []
Test ICs : []

Saved:
/content/drive/MyDrive/AI_4DVAR/klein_ml_1L/dataset_train_1L.npz
/content/drive/MyDrive/AI_4DVAR/klein_ml_1L/dataset_val_1L.npz
/content/drive/MyDrive/AI_4DVAR/klein_ml_1L/dataset_test_1L.npz
/content/drive/MyDrive/AI_4DVAR/klein_ml_1L/dataset_summary.txt

Shapes:
train X: (Empty) train Y: (Empty)
val   X: (Empty) val   Y: (Empty)
test  X: (Empty) test  Y: (Empty)


# train_cnn_emulator

In [ ]:
# ============================================================
# train_cnn_emulator_1layer.py
# ------------------------------------------------------------
# CNN emulator for 1-layer shallow-water forecasting on
# Klein/twisted beta-plane.
#
# Input channels:
#   [eta, uc, vc, f, y_norm]
#
# Target channels:
#   [eta, uc, vc] at next saved time
#
# Saves:
#   best_model.pt
#   norm_stats.npz
#   loss_history.csv
#   loss_curve.png
#   prediction plots
#
# Colab usage:
# !python /content/train_cnn_emulator_1layer.py
# ============================================================

import os
import csv
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from google.colab import drive

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# -------------------------
# Paths
# -------------------------
# drive.mount("/content/drive") # This line is redundant as drive is already mounted.
ROOT_DATA = "/content/drive/MyDrive/AI_4DVAR/klein_ml_1L"
OUT_DIR   = "/content/drive/MyDrive/AI_4DVAR/klein_ml_1L_ckpt_cnn"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(ROOT_DATA, "dataset_train_1L.npz")
VAL_PATH   = os.path.join(ROOT_DATA, "dataset_val_1L.npz")
TEST_PATH  = os.path.join(ROOT_DATA, "dataset_test_1L.npz")

# -------------------------
# Config
# -------------------------
class CFG:
    epochs = 200
    batch_size = 8
    lr = 2e-3
    weight_decay = 1e-6
    print_every = 10

    # loss weights
    lambda_eta = 1.0
    lambda_uv  = 1.0
    lambda_mass = 1e-2   # small eta-domain-mean penalty

    # model
    width = 48
    grad_clip = 1.0

    # plots
    n_plot = 4

cfg = CFG()

# -------------------------
# Dataset loader
# -------------------------
def load_dataset(path):
    d = np.load(path, allow_pickle=True)
    return {
        "X": d["X"].astype(np.float32),           # (N, 5, ny, nx)
        "Y": d["Y"].astype(np.float32),           # (N, 3, ny, nx)
        "ic_key": d["ic_key"],
        "step0": d["step0"].astype(np.int32),
        "step1": d["step1"].astype(np.int32),
    }

train_data = load_dataset(TRAIN_PATH)
val_data   = load_dataset(VAL_PATH)
test_data  = load_dataset(TEST_PATH)

print("Train X:", train_data["X"].shape, "Y:", train_data["Y"].shape)
print("Val   X:", val_data["X"].shape,   "Y:", val_data["Y"].shape)
print("Test  X:", test_data["X"].shape,  "Y:", test_data["Y"].shape)

# -------------------------
# Normalization
# -------------------------
def compute_channel_stats(X, Y):
    """
    X shape: (N, 5, ny, nx)
    Y shape: (N, 3, ny, nx)
    Compute per-channel mean/std from TRAIN set only.
    """
    x_mean = X.mean(axis=(0,2,3), keepdims=True)
    x_std  = X.std(axis=(0,2,3), keepdims=True) + 1e-6
    y_mean = Y.mean(axis=(0,2,3), keepdims=True)
    y_std  = Y.std(axis=(0,2,3), keepdims=True) + 1e-6
    return x_mean.astype(np.float32), x_std.astype(np.float32), y_mean.astype(np.float32), y_std.astype(np.float32)

x_mean, x_std, y_mean, y_std = compute_channel_stats(train_data["X"], train_data["Y"])

np.savez_compressed(
    os.path.join(OUT_DIR, "norm_stats.npz"),
    x_mean=x_mean, x_std=x_std,
    y_mean=y_mean, y_std=y_std,
)
print("Saved normalization stats.")

def normalize_X(X):
    return (X - x_mean) / x_std

def normalize_Y(Y):
    return (Y - y_mean) / y_std

def denormalize_Y(Yn):
    return Yn * y_std + y_mean

train_X = normalize_X(train_data["X"])
train_Y = normalize_Y(train_data["Y"])

val_X = normalize_X(val_data["X"])
val_Y = normalize_Y(val_data["Y"])

test_X = normalize_X(test_data["X"])
test_Y = normalize_Y(test_data["Y"])

# -------------------------
# Torch dataset
# -------------------------
class ArrayDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y, meta=None):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)
        self.meta = meta

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        return self.X[i], self.Y[i]

train_ds = ArrayDataset(train_X, train_Y)
val_ds   = ArrayDataset(val_X, val_Y)
test_ds  = ArrayDataset(test_X, test_Y)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0
)
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0
)

# -------------------------
# Model
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1),
            nn.GELU(),
        )
    def forward(self, x):
        return self.block(x)

class CNNEmulator(nn.Module):
    """
    Compact residual CNN:
      [eta, uc, vc, f, y_norm] -> [eta, uc, vc] at next time
    Predicts normalized target directly.
    """
    def __init__(self, cin=5, cout=3, width=48):
        super().__init__()
        self.in_proj = nn.Conv2d(cin, width, 3, padding=1)
        self.b1 = ConvBlock(width, width)
        self.b2 = ConvBlock(width, width)
        self.b3 = ConvBlock(width, width)
        self.out_proj = nn.Conv2d(width, cout, 1)

    def forward(self, x):
        z = self.in_proj(x)
        z = z + self.b1(z)
        z = z + self.b2(z)
        z = z + self.b3(z)
        y = self.out_proj(z)
        return y

model = CNNEmulator(width=cfg.width).to(device)

# -------------------------
# Loss
# -------------------------
def loss_fn(pred_n, targ_n):
    """
    pred_n, targ_n are normalized outputs.
    Weighted channel loss + mass penalty on denormalized eta.
    """
    # channel-wise MSE in normalized space
    diff = pred_n - targ_n
    loss_eta = torch.mean(diff[:, 0:1]**2)
    loss_uv  = torch.mean(diff[:, 1:3]**2)

    # denormalize eta for mass-like mean penalty
    y_mean_t = torch.tensor(y_mean, device=pred_n.device)
    y_std_t  = torch.tensor(y_std,  device=pred_n.device)

    pred_eta = pred_n[:, 0:1] * y_std_t[:, 0:1] + y_mean_t[:, 0:1]
    targ_eta = targ_n[:, 0:1] * y_std_t[:, 0:1] + y_mean_t[:, 0:1]

    mass_pen = torch.mean((pred_eta.mean(dim=(-2,-1)) - targ_eta.mean(dim=(-2,-1)))**2)

    loss = (
        cfg.lambda_eta * loss_eta +
        cfg.lambda_uv  * loss_uv  +
        cfg.lambda_mass * mass_pen
    )
    return loss, {
        "eta": loss_eta.detach().item(),
        "uv": loss_uv.detach().item(),
        "mass": mass_pen.detach().item(),
    }

# -------------------------
# Optimizer
# -------------------------
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=15
)

# -------------------------
# Training / eval loops
# -------------------------
def run_epoch(loader, training):
    if training:
        model.train()
    else:
        model.eval()

    loss_sum = 0.0
    eta_sum = 0.0
    uv_sum = 0.0
    mass_sum = 0.0
    n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        with torch.set_grad_enabled(training):
            pred = model(xb)
            loss, comps = loss_fn(pred, yb)

            if training:
                if not torch.isfinite(loss):
                    optimizer.zero_grad(set_to_none=True)
                    continue
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                optimizer.step()

        bs = xb.shape[0]
        loss_sum += loss.detach().item() * bs
        eta_sum  += comps["eta"]  * bs
        uv_sum   += comps["uv"]   * bs
        mass_sum += comps["mass"] * bs
        n += bs

    return {
        "loss": loss_sum / max(n, 1),
        "eta": eta_sum / max(n, 1),
        "uv": uv_sum / max(n, 1),
        "mass": mass_sum / max(n, 1),
    }

# -------------------------
# Train
# -------------------------
history = []
best_val = float("inf")
best_path = os.path.join(OUT_DIR, "best_model.pt")

for ep in range(cfg.epochs):
    tr = run_epoch(train_loader, training=True)
    va = run_epoch(val_loader, training=False)

    scheduler.step(va["loss"])

    history.append({
        "epoch": ep,
        "train": tr["loss"],
        "val": va["loss"],
        "train_eta": tr["eta"],
        "train_uv": tr["uv"],
        "train_mass": tr["mass"],
        "val_eta": va["eta"],
        "val_uv": va["uv"],
        "val_mass": va["mass"],
        "lr": optimizer.param_groups[0]["lr"],
    })

    if va["loss"] < best_val:
        best_val = va["loss"]
        torch.save(model.state_dict(), best_path)

    if (ep % cfg.print_every) == 0 or ep == cfg.epochs - 1:
        print(
            f"[ep {ep:03d}] "
            f"train={tr['loss']:.4e} val={va['loss']:.4e} "
            f"(eta tr/va={tr['eta']:.3e}/{va['eta']:.3e}, "
            f"uv tr/va={tr['uv']:.3e}/{va['uv']:.3e}, "
            f"mass tr/va={tr['mass']:.3e}/{va['mass']:.3e}) "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

print("Best val loss:", best_val)
print("Saved:", best_path)

# -------------------------
# Save history
# -------------------------
hist_csv = os.path.join(OUT_DIR, "loss_history.csv")
with open(hist_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    w.writeheader()
    w.writerows(history)
print("Saved:", hist_csv)

# plot loss
plt.figure(figsize=(7,4))
plt.plot([h["epoch"] for h in history], [h["train"] for h in history], label="train")
plt.plot([h["epoch"] for h in history], [h["val"] for h in history], label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("CNN emulator loss")
plt.tight_layout()
loss_png = os.path.join(OUT_DIR, "loss_curve.png")
plt.savefig(loss_png, dpi=140)
plt.close()
print("Saved:", loss_png)

# -------------------------
# Reload best and evaluate
# -------------------------
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

def predict_dataset(Xn):
    preds = []
    with torch.no_grad():
        for i in range(0, len(Xn), cfg.batch_size):
            xb = torch.from_numpy(Xn[i:i+cfg.batch_size]).to(device)
            yp = model(xb).cpu().numpy()
            preds.append(yp)
    return np.concatenate(preds, axis=0)

val_pred_n  = predict_dataset(val_X)
test_pred_n = predict_dataset(test_X)

val_pred  = denormalize_Y(val_pred_n)
test_pred = denormalize_Y(test_pred_n)

val_true  = val_data["Y"]
test_true = test_data["Y"]

# -------------------------
# Metrics
# -------------------------
def channel_rmse(pred, true):
    out = {}
    names = ["eta", "uc", "vc"]
    for i, name in enumerate(names):
        out[name] = float(np.sqrt(np.mean((pred[:, i] - true[:, i])**2)))
    return out

val_rmse = channel_rmse(val_pred, val_true)
test_rmse = channel_rmse(test_pred, test_true)

print("\nValidation RMSE:", val_rmse)
print("Test RMSE:", test_rmse)

with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write("Validation RMSE\n")
    for k, v in val_rmse.items():
        f.write(f"{k}: {v:.6e}\n")
    f.write("\nTest RMSE\n")
    for k, v in test_rmse.items():
        f.write(f"{k}: {v:.6e}\n")
print("Saved metrics.txt")

# -------------------------
# Prediction plots
# -------------------------
plot_dir = os.path.join(OUT_DIR, "predictions")
os.makedirs(plot_dir, exist_ok=True)

def plot_sample(idx, pred, true, split_name, meta):
    names = ["eta", "uc", "vc"]
    fig, axs = plt.subplots(3, 3, figsize=(11, 10))

    for i, nm in enumerate(names):
        im = axs[i,0].imshow(true[i], origin="lower")
        axs[i,0].set_title(f"Truth {nm}")
        plt.colorbar(im, ax=axs[i,0], fraction=0.046, pad=0.04)

        im = axs[i,1].imshow(pred[i], origin="lower")
        axs[i,1].set_title(f"Pred {nm}")
        plt.colorbar(im, ax=axs[i,1], fraction=0.046, pad=0.04)

        err = pred[i] - true[i]
        im = axs[i,2].imshow(err, origin="lower")
        axs[i,2].set_title(f"Error {nm}")
        plt.colorbar(im, ax=axs[i,2], fraction=0.046, pad=0.04)

    ic = meta["ic_key"][idx]
    s0 = int(meta["step0"][idx])
    s1 = int(meta["step1"][idx])
    plt.suptitle(f"{split_name}: {ic}  {s0}->{s1}", y=0.995)
    plt.tight_layout()
    out = os.path.join(plot_dir, f"{split_name}_{idx:03d}_{ic}_{s0}_{s1}.png")
    plt.savefig(out, dpi=140)
    plt.close()

# choose a few val and test samples
for i in range(min(cfg.n_plot, len(val_pred))):
    plot_sample(i, val_pred[i], val_true[i], "val", val_data)

for i in range(min(cfg.n_plot, len(test_pred))):
    plot_sample(i, test_pred[i], test_true[i], "test", test_data)

print("Saved prediction plots in:", plot_dir)

# -------------------------
# Optional quick rollout-like check on one trajectory
# -------------------------
def group_by_ic(meta):
    out = {}
    for i, ic in enumerate(meta["ic_key"]):
        out.setdefault(str(ic), []).append(i)
    return out

def rollout_check(meta, Xn, Ytrue, split_name):
    """
    Very simple chained prediction across the 6 macro steps of one IC.
    Only uses learned model recursively.
    Uses f and y_norm from input sample, and predicted [eta,uc,vc] becomes next state's first 3 channels.
    """
    groups = group_by_ic(meta)
    ic = sorted(groups.keys())[0]
    idxs = sorted(groups[ic], key=lambda j: meta["step0"][j])

    # start from first X
    x = Xn[idxs[0]:idxs[0]+1].copy()  # normalized
    traj_pred = []
    traj_true = []

    with torch.no_grad():
        for j in idxs:
            # predict next
            xb = torch.from_numpy(x).to(device)
            yp_n = model(xb).cpu().numpy()[0]
            yp = denormalize_Y(yp_n[None])[0]

            traj_pred.append(yp)
            traj_true.append(Ytrue[j])

            # build next input:
            # replace [eta,uc,vc] with predicted, keep f,y_norm from actual next-input sample
            if j != idxs[-1]:
                x_next = Xn[j+1:j+2].copy()
                # normalize predicted output back into first 3 channels
                yp_n3 = yp_n
                x_next[0, 0:3] = yp_n3
                x = x_next

    traj_pred = np.stack(traj_pred, axis=0)
    traj_true = np.stack(traj_true, axis=0)

    # eta rmse per lead
    eta_rmse = [math.sqrt(np.mean((traj_pred[k,0] - traj_true[k,0])**2)) for k in range(len(traj_pred))]
    plt.figure(figsize=(6,4))
    plt.plot(range(1, len(eta_rmse)+1), eta_rmse, "-o")
    plt.xlabel("macro lead step")
    plt.ylabel("RMSE eta")
    plt.grid(True, alpha=0.3)
    plt.title(f"{split_name} rollout-like check: {ic}")
    out = os.path.join(OUT_DIR, f"rollout_like_{split_name}_{ic}.png")
    plt.tight_layout()
    plt.savefig(out, dpi=140)
    plt.close()
    print(f"Saved rollout-like check: {out}")

rollout_check(val_data, val_X, val_true, "val")
rollout_check(test_data, test_X, test_true, "test")

print("\nDone.")

Train X: (114, 5, 128, 256) Y: (114, 3, 128, 256)
Val   X: (30, 5, 128, 256) Y: (30, 3, 128, 256)
Test  X: (24, 5, 128, 256) Y: (24, 3, 128, 256)
Saved normalization stats.
[ep 000] train=1.0540e+00 val=1.0183e+00 (eta tr/va=5.110e-01/6.754e-01, uv tr/va=2.573e-01/3.165e-01, mass tr/va=2.856e+01/2.644e+00) lr=2.00e-03
[ep 010] train=1.6975e-01 val=2.2560e-01 (eta tr/va=9.863e-02/1.415e-01, uv tr/va=5.177e-02/6.506e-02, mass tr/va=1.934e+00/1.908e+00) lr=2.00e-03
[ep 020] train=1.0912e-01 val=5.1859e-01 (eta tr/va=6.183e-02/1.599e-01, uv tr/va=3.444e-02/6.204e-02, mass tr/va=1.286e+00/2.966e+01) lr=2.00e-03
[ep 030] train=1.0893e-01 val=2.0174e-01 (eta tr/va=5.267e-02/7.866e-02, uv tr/va=2.268e-02/3.787e-02, mass tr/va=3.358e+00/8.521e+00) lr=2.00e-03
[ep 040] train=7.7436e-02 val=1.3258e-01 (eta tr/va=3.705e-02/7.765e-02, uv tr/va=2.489e-02/3.462e-02, mass tr/va=1.550e+00/2.032e+00) lr=2.00e-03
[ep 050] train=6.5731e-02 val=1.0963e-01 (eta tr/va=3.295e-02/5.793e-02, uv tr/va=1.721e-02/

# Evaluation Script

In [ ]:
# ==============================================
# eval_cnn_emulator_1layer.py
# Corrected version for datasets with:
# X, Y, ic_key, step0, step1
# ==============================================

import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_TEST = f"{ROOT}/klein_ml_1L/dataset_test_1L.npz"
DATA_VAL  = f"{ROOT}/klein_ml_1L/dataset_val_1L.npz"

MODEL_PATH = f"{ROOT}/klein_ml_1L_ckpt_cnn/best_model.pt"
OUT_DIR    = f"{ROOT}/klein_ml_1L_ckpt_cnn"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1),
            nn.GELU(),
        )
    def forward(self, x):
        return self.block(x)

class CNNEmulator(nn.Module):
    def __init__(self, cin=5, cout=3, width=48):
        super().__init__()
        self.in_proj = nn.Conv2d(cin, width, 3, padding=1)
        self.b1 = ConvBlock(width, width)
        self.b2 = ConvBlock(width, width)
        self.b3 = ConvBlock(width, width)
        self.out_proj = nn.Conv2d(width, cout, 1)

    def forward(self, x):
        z = self.in_proj(x)
        z = z + self.b1(z)
        z = z + self.b2(z)
        z = z + self.b3(z)
        return self.out_proj(z)

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d["X"], d["Y"], d["ic_key"], d["step0"], d["step1"]

X_val, Y_val, ic_val, step0_val, step1_val = load_npz(DATA_VAL)
X_test, Y_test, ic_test, step0_test, step1_test = load_npz(DATA_TEST)

print("Val:", X_val.shape, Y_val.shape)
print("Test:", X_test.shape, Y_test.shape)

model = CNNEmulator().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

def evaluate_one_step(X, Y):
    with torch.no_grad():
        x = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        y_true = torch.tensor(Y, dtype=torch.float32).to(DEVICE)
        y_pred = model(x)
    y_pred = y_pred.cpu().numpy()
    y_true = y_true.cpu().numpy()

    out = {
        "eta": rmse(y_pred[:, 0], y_true[:, 0]),
        "uc":  rmse(y_pred[:, 1], y_true[:, 1]),
        "vc":  rmse(y_pred[:, 2], y_true[:, 2]),
    }
    return out, y_pred

rmse_val, pred_val = evaluate_one_step(X_val, Y_val)
rmse_test, pred_test = evaluate_one_step(X_test, Y_test)

print("\nValidation RMSE:", rmse_val)
print("Test RMSE:", rmse_test)

with open(f"{OUT_DIR}/metrics.txt", "w") as f:
    f.write(f"Validation RMSE: {rmse_val}\n")
    f.write(f"Test RMSE: {rmse_test}\n")

PLOT_DIR = os.path.join(OUT_DIR, "predictions")
os.makedirs(PLOT_DIR, exist_ok=True)

def plot_sample(y_true, y_pred, title, out_png):
    fields = ["eta", "uc", "vc"]
    plt.figure(figsize=(12, 6))
    for i in range(3):
        plt.subplot(3, 3, i*3+1)
        plt.imshow(y_true[i], origin="lower")
        plt.title(f"{fields[i]} truth")
        plt.colorbar()

        plt.subplot(3, 3, i*3+2)
        plt.imshow(y_pred[i], origin="lower")
        plt.title(f"{fields[i]} pred")
        plt.colorbar()

        plt.subplot(3, 3, i*3+3)
        plt.imshow(y_pred[i] - y_true[i], origin="lower")
        plt.title(f"{fields[i]} error")
        plt.colorbar()
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=120)
    plt.close()

for i in range(min(5, len(X_test))):
    title = f"test {ic_test[i]} {step0_test[i]}->{step1_test[i]}"
    plot_sample(Y_test[i], pred_test[i], title, f"{PLOT_DIR}/test_{i}.png")

ROLL_DIR = os.path.join(OUT_DIR, "rollout_like")
os.makedirs(ROLL_DIR, exist_ok=True)

def normalize_X(X, x_mean, x_std):
    return (X - x_mean) / x_std

# load norm stats
norm = np.load(f"{OUT_DIR}/norm_stats.npz")
x_mean = norm["x_mean"].astype(np.float32)
x_std  = norm["x_std"].astype(np.float32)
y_mean = norm["y_mean"].astype(np.float32)
y_std  = norm["y_std"].astype(np.float32)

def denormalize_Y(Yn):
    return Yn * y_std + y_mean

def group_by_ic(ic_keys):
    out = {}
    for i, ic in enumerate(ic_keys):
        out.setdefault(str(ic), []).append(i)
    return out

def rollout_one_ic(X_raw_ic):
    x_cur_raw = X_raw_ic[0:1].copy()
    preds = []
    with torch.no_grad():
        for _ in range(len(X_raw_ic)):
            x_cur = normalize_X(x_cur_raw, x_mean, x_std)
            xb = torch.tensor(x_cur, dtype=torch.float32).to(DEVICE)
            yp_n = model(xb).cpu().numpy()
            yp = denormalize_Y(yp_n)[0]
            preds.append(yp)

            # next input: replace eta,uc,vc with predicted values
            if len(preds) < len(X_raw_ic):
                x_next_raw = X_raw_ic[len(preds):len(preds)+1].copy()
                x_next_raw[0, 0:3] = yp
                x_cur_raw = x_next_raw
    return np.array(preds)

def evaluate_rollout(X, Y, ic_keys, step0, tag):
    groups = group_by_ic(ic_keys)
    results = []

    for ic in sorted(groups.keys()):
        idx = sorted(groups[ic], key=lambda j: step0[j])
        X_ic = X[idx]
        Y_ic = Y[idx]

        pred = rollout_one_ic(X_ic)
        truth = Y_ic[:len(pred)]

        eta_rmse = [rmse(pred[k, 0], truth[k, 0]) for k in range(len(pred))]
        uc_rmse  = [rmse(pred[k, 1], truth[k, 1]) for k in range(len(pred))]
        vc_rmse  = [rmse(pred[k, 2], truth[k, 2]) for k in range(len(pred))]

        plt.figure(figsize=(7,4))
        plt.plot(eta_rmse, "-o", label="eta")
        plt.plot(uc_rmse, "-s", label="uc")
        plt.plot(vc_rmse, "-^", label="vc")
        plt.xlabel("macro lead step")
        plt.ylabel("RMSE")
        plt.title(f"{tag} rollout-like: {ic}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{ROLL_DIR}/rollout_{tag}_{ic}.png", dpi=120)
        plt.close()

        results.append(
            f"{tag:5s} {ic:16s}  final_eta_rmse={eta_rmse[-1]:.6e}  "
            f"final_uc_rmse={uc_rmse[-1]:.6e}  final_vc_rmse={vc_rmse[-1]:.6e}"
        )

    return results

roll_val = evaluate_rollout(X_val, Y_val, ic_val, step0_val, "val")
roll_test = evaluate_rollout(X_test, Y_test, ic_test, step0_test, "test")

with open(f"{OUT_DIR}/rollout_like_summary.txt", "w") as f:
    f.write("VAL:\n")
    for line in roll_val:
        f.write(line + "\n")
    f.write("\nTEST:\n")
    for line in roll_test:
        f.write(line + "\n")

print("Saved rollout_like_summary.txt")
print("Done.")

Val: (30, 5, 128, 256) (30, 3, 128, 256)
Test: (24, 5, 128, 256) (24, 3, 128, 256)

Validation RMSE: {'eta': 28.051794052124023, 'uc': 6.377634048461914, 'vc': 18.489965438842773}
Test RMSE: {'eta': 86.70093536376953, 'uc': 20.659069061279297, 'vc': 48.13201904296875}
Saved rollout_like_summary.txt
Done.


# Multistep Training

In [ ]:
# ============================================================
# run_fd_1layer_catalog.py
# ------------------------------------------------------------
# Run 1-layer shallow-water FD model on twisted/Klein beta-plane
# for every IC in a C-grid bundle.
#
# INPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz
#
# OUTPUT:
#   /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/<IC_KEY>/klein_step_XXXXXX.npz
#   plus plots and time series
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Paths
# -------------------------
IC_BUNDLE = "/content/drive/MyDrive/AI_4DVAR/klein_ics_1L/bundle_cgrid_1L.npz"
ROOT_OUT  = "/content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L"

# -------------------------
# Physics / grid
# -------------------------
g = 9.81
H = 1000.0
nx = 256
ny = 128
Lx = 2.0e7
Ly = 8.0e6
dx = Lx / nx
dy = Ly / ny
fp = 8.0e-5

dt = 30.0
nt = 1200
SAVE_STEPS = {0, 200, 400, 600, 800, 1000, 1200}

# diffusion
nu2_u, nu2_v, nu2_h = 1.0e4, 1.0e4, 5.0e3
nu4_u, nu4_v, nu4_h = 5.0e10, 5.0e10, 2.5e10

HMIN = 0.5

# -------------------------
# Geometry
# -------------------------
x_c = np.linspace(0.5*dx, Lx-0.5*dx, nx)
y_c = np.linspace(0.5*dy, Ly-0.5*dy, ny)
Xc, Yc = np.meshgrid(x_c, y_c)
phi_c = np.pi*((Yc/Ly) - 0.5)
f_c = fp*np.sin(phi_c)

x_u = np.linspace(0.0, Lx, nx+1)
y_v = np.linspace(0.0, Ly, ny+1)
Xu, Yu = np.meshgrid(x_u, y_c)
Xv, Yv = np.meshgrid(x_c, y_v)
phi_u = np.pi*((Yu/Ly) - 0.5)
phi_v = np.pi*((Yv/Ly) - 0.5)
f_u = fp*np.sin(phi_u)
f_v = fp*np.sin(phi_v)

# -------------------------
# Klein twist BCs
# -------------------------
def twist_reflect_x(arr):
    return arr[..., ::-1]

def apply_bc_1l(u, v, h):
    # centers even
    h[0, :]  = 0.5*(h[1, :]  + twist_reflect_x(h[1, :]))
    h[-1, :] = 0.5*(h[-2, :] + twist_reflect_x(h[-2, :]))
    # u odd
    u[0, :]  = 0.5*(u[1, :]  - twist_reflect_x(u[1, :]))
    u[-1, :] = 0.5*(u[-2, :] - twist_reflect_x(u[-2, :]))
    # v even
    v[0, :]  = 0.5*(v[1, :]  + twist_reflect_x(v[1, :]))
    v[-1, :] = 0.5*(v[-2, :] + twist_reflect_x(v[-2, :]))
    return u, v, h

# -------------------------
# C-grid helpers
# -------------------------
def center_from_u(u):
    return 0.5*(u[:, :-1] + u[:, 1:])

def center_from_v(v):
    return 0.5*(v[:-1, :] + v[1:, :])

def avg_x(a):
    return 0.5*(np.pad(a, ((0,0),(1,0)), mode='wrap') + np.pad(a, ((0,0),(0,1)), mode='wrap'))

def avg_y(a):
    return 0.5*(np.pad(a, ((1,0),(0,0)), mode='edge') + np.pad(a, ((0,1),(0,0)), mode='edge'))

def ddx_c_to_u(phi):
    L = np.pad(phi, ((0,0),(1,0)), mode='wrap')
    R = np.pad(phi, ((0,0),(0,1)), mode='wrap')
    return (R - L) / (2.0*dx)

def ddy_c_to_v(phi):
    T = np.pad(phi, ((1,0),(0,0)), mode='edge')
    B = np.pad(phi, ((0,1),(0,0)), mode='edge')
    return (B - T) / (2.0*dy)

def ddx_u_to_c(phi_u):
    return (phi_u[:, 1:] - phi_u[:, :-1]) / dx

def ddy_v_to_c(phi_v):
    return (phi_v[1:, :] - phi_v[:-1, :]) / dy

# -------------------------
# Laplacians
# -------------------------
def lap_u(u):
    ue = np.pad(u, ((0,0),(1,1)), mode='wrap')
    u_xx = (ue[:, :-2] - 2*ue[:, 1:-1] + ue[:, 2:]) / dx**2
    ue2 = np.pad(u, ((1,1),(0,0)), mode='edge')
    u_yy = (ue2[:-2, :] - 2*ue2[1:-1, :] + ue2[2:, :]) / dy**2
    return u_xx + u_yy

def lap_v(v):
    ve = np.pad(v, ((0,0),(1,1)), mode='wrap')
    v_xx = (ve[:, :-2] - 2*ve[:, 1:-1] + ve[:, 2:]) / dx**2
    ve2 = np.pad(v, ((1,1),(0,0)), mode='edge')
    v_yy = (ve2[:-2, :] - 2*ve2[1:-1, :] + ve2[2:, :]) / dy**2
    return v_xx + v_yy

def lap_c(h):
    he = np.pad(h, ((0,0),(1,1)), mode='wrap')
    h_xx = (he[:, :-2] - 2*he[:, 1:-1] + he[:, 2:]) / dx**2
    he2 = np.pad(h, ((1,1),(0,0)), mode='edge')
    h_yy = (he2[:-2, :] - 2*he2[1:-1, :] + he2[2:, :]) / dy**2
    return h_xx + h_yy

def bih_u(u): return lap_u(lap_u(u))
def bih_v(v): return lap_v(lap_v(v))
def bih_c(h): return lap_c(lap_c(h))

# -------------------------
# Vorticity
# -------------------------
def compute_corner_vort(u, v):
    v_w = np.pad(v, ((0,0),(1,0)), mode='wrap')
    v_e = np.pad(v, ((0,0),(0,1)), mode='wrap')
    dv_dx = (v_e - v_w) / (2*dx)

    u_s = np.pad(u, ((1,0),(0,0)), mode='edge')
    u_n = np.pad(u, ((0,1),(0,0)), mode='edge')
    du_dy = (u_n - u_s) / (2*dy)

    return dv_dx - du_dy

def to_u_from_corners(a):
    return 0.5*(a[:-1, :] + a[1:, :])

def to_v_from_corners(a):
    return 0.5*(a[:, :-1] + a[:, 1:])

# -------------------------
# Positivity floor
# -------------------------
def enforce_floor_ke_preserving(u, v, h, hmin=HMIN):
    mask = (h < hmin)
    if not np.any(mask):
        return u, v, h, 0

    s_c = np.ones_like(h, dtype=np.float32)
    s_c[mask] = np.sqrt(np.maximum(h[mask], 1e-12) / hmin)

    s_u = 0.5*(np.pad(s_c, ((0,0),(1,0)), mode='wrap') + np.pad(s_c, ((0,0),(0,1)), mode='wrap'))
    s_v = 0.5*(np.pad(s_c, ((1,0),(0,0)), mode='edge') + np.pad(s_c, ((0,1),(0,0)), mode='edge'))

    u = u * s_u
    v = v * s_v
    h = np.maximum(h, hmin)
    return u, v, h, int(mask.sum())

# -------------------------
# RHS (1-layer AL-style)
# -------------------------
def rhs_1l(u, v, h):
    u, v, h = apply_bc_1l(u, v, h)

    uc = center_from_u(u)
    vc = center_from_v(v)
    K = 0.5*(uc**2 + vc**2)

    eta = h - H
    Phi = g * eta

    dPhidx_u = ddx_c_to_u(Phi)
    dPhidy_v = ddy_c_to_v(Phi)
    dKdx_u   = ddx_c_to_u(K)
    dKdy_v   = ddy_c_to_v(K)

    z_corners = compute_corner_vort(u, v)
    eta_u = to_u_from_corners(z_corners) + f_u
    eta_v = to_v_from_corners(z_corners) + f_v

    v_tu = avg_x(center_from_v(v))
    u_tv = avg_y(center_from_u(u))

    du = -(dPhidx_u + dKdx_u) + eta_u * v_tu + nu2_u*lap_u(u) + nu4_u*bih_u(u)
    dv = -(dPhidy_v + dKdy_v) - eta_v * u_tv + nu2_v*lap_v(v) + nu4_v*bih_v(v)

    h_u = avg_x(h)
    h_v = avg_y(h)
    F_u = h_u * u
    F_v = h_v * v

    dhdt = -(ddx_u_to_c(F_u) + ddy_v_to_c(F_v)) + nu2_h*lap_c(h) + nu4_h*bih_c(h)

    return apply_bc_1l(du, dv, dhdt)

# -------------------------
# RK4
# -------------------------
def rk4_1l(u, v, h, dt):
    k1u, k1v, k1h = rhs_1l(u, v, h)

    ub, vb, hb = apply_bc_1l(u + 0.5*dt*k1u, v + 0.5*dt*k1v, h + 0.5*dt*k1h)
    k2u, k2v, k2h = rhs_1l(ub, vb, hb)

    uc, vc, hc = apply_bc_1l(u + 0.5*dt*k2u, v + 0.5*dt*k2v, h + 0.5*dt*k2h)
    k3u, k3v, k3h = rhs_1l(uc, vc, hc)

    ud, vd, hd = apply_bc_1l(u + dt*k3u, v + dt*k3v, h + dt*k3h)
    k4u, k4v, k4h = rhs_1l(ud, vd, hd)

    u_new = u + (dt/6.0)*(k1u + 2*k2u + 2*k3u + k4u)
    v_new = v + (dt/6.0)*(k1v + 2*k2v + 2*k3v + k4v) # Corrected: k4h -> k4v
    h_new = h + (dt/6.0)*(k1h + 2*k2h + 2*k3h + k4h)

    return apply_bc_1l(u_new, v_new, h_new)

# -------------------------
# Diagnostics / save
# -------------------------
def total_mass(h):
    return float(np.sum(h) * dx * dy)

def total_energy(u, v, h):
    uc = center_from_u(u)
    vc = center_from_v(v)
    eta = h - H
    ke = 0.5*h*(uc**2 + vc**2)
    pe = 0.5*g*(eta**2)
    return float(np.sum((ke + pe) * dx * dy))

def save_centered_1L(ic_key, step, u, v, h, t):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    os.makedirs(ic_dir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    path = os.path.join(ic_dir, f"klein_step_{step:06d}.npz")
    np.savez_compressed(
        path,
        eta=eta.astype(np.float32),
        uc=uc.astype(np.float32),
        vc=vc.astype(np.float32),
        h=h.astype(np.float32),
        f=f_c.astype(np.float32),
        y_m=y_c.astype(np.float32),
        H=np.float32(H),
        dt=np.float32(dt),
        t=np.float32(t),
        nx=np.int32(nx), ny=np.int32(ny),
        dx=np.float32(dx), dy=np.float32(dy), fp=np.float32(fp),
    )
    return path

def quick_plot_1L(ic_key, step, u, v, h):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    eta = h - H
    uc = center_from_u(u)
    vc = center_from_v(v)

    xkm = Xc / 1e3
    ykm = (Yc - 0.5*Ly) / 1e3

    fig, axs = plt.subplots(1, 3, figsize=(14, 3.5))
    im = axs[0].pcolormesh(xkm, ykm, eta, shading="auto")
    axs[0].set_title("eta (m)")
    plt.colorbar(im, ax=axs[0])

    im = axs[1].pcolormesh(xkm, ykm, uc, shading="auto")
    axs[1].set_title("u_c (m/s)")
    plt.colorbar(im, ax=axs[1])

    im = axs[2].pcolormesh(xkm, ykm, vc, shading="auto")
    axs[2].set_title("v_c (m/s)")
    plt.colorbar(im, ax=axs[2])

    plt.tight_layout()
    plt.savefig(os.path.join(pdir, f"fields_step_{step:06d}.png"), dpi=120)
    plt.close()

def plot_mass_energy(ic_key, steps, m_series, e_series):
    ic_dir = os.path.join(ROOT_OUT, ic_key)
    pdir = os.path.join(ic_dir, "plots")
    os.makedirs(pdir, exist_ok=True)

    tdays = np.asarray(steps) * dt / 86400.0
    M0 = m_series[0]
    E0 = e_series[0]

    plt.figure(figsize=(8,4))
    plt.plot(tdays, (np.asarray(m_series)-M0)/M0, label="ΔM/M0")
    plt.plot(tdays, (np.asarray(e_series)-E0)/E0, label="ΔE/E0")
    plt.xlabel("time (days)")
    plt.ylabel("relative change")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(pdir, "mass_energy_timeseries.png"), dpi=120)
    plt.close()

# -------------------------
# Load bundle and IC keys
# -------------------------
def list_ic_keys(bundle_path):
    d = np.load(bundle_path)
    keys = sorted(set(k[:-2] for k in d.files if k.endswith("_h")))
    return keys

def load_ic(bundle_path, ic_key):
    d = np.load(bundle_path)
    h = d[f"{ic_key}_h"].astype(np.float32)
    u = d[f"{ic_key}_u"].astype(np.float32)
    v = d[f"{ic_key}_v"].astype(np.float32)
    return h, u, v

# -------------------------
# Run one IC
# -------------------------
def run_ic(ic_key):
    print(f"\n=== 1L IC: {ic_key} | nx={nx}, ny={ny}, dt={dt:.1f}s, nt={nt} ===")
    h, u, v = load_ic(IC_BUNDLE, ic_key)
    u, v, h = apply_bc_1l(u, v, h)
    h[:] = np.maximum(h, 5.0)

    steps = []
    m_series = []
    e_series = []

    M0 = total_mass(h)
    E0 = total_energy(u, v, h)
    t = 0.0

    steps.append(0)
    m_series.append(M0)
    e_series.append(E0)

    if 0 in SAVE_STEPS:
        p = save_centered_1L(ic_key, 0, u, v, h, t)
        quick_plot_1L(ic_key, 0, u, v, h)
        print(f"[save] {ic_key} step=0 -> {p}")

    for n in range(1, nt+1):
        u, v, h = rk4_1l(u, v, h, dt)
        t += dt

        u, v, h, _ = enforce_floor_ke_preserving(u, v, h, HMIN)

        steps.append(n)
        m_series.append(total_mass(h))
        e_series.append(total_energy(u, v, h))

        if n in SAVE_STEPS:
            p = save_centered_1L(ic_key, n, u, v, h, t)
            quick_plot_1L(ic_key, n, u, v, h)
            print(f"[save] {ic_key} step={n} -> {p}")

        if (n % 100) == 0 or n == 1:
            uc = center_from_u(u)
            vc = center_from_v(v)
            umax = float(np.max(np.sqrt(uc*uc + vc*vc)))
            dM = (m_series[-1] - M0) / M0
            dE = (e_series[-1] - E0) / E0
            print(f"[{n:5d}] dM/M0={dM:+.3e}  dE/E0={dE:+.3e}  umax={umax:6.2f}")

    plot_mass_energy(ic_key, steps, m_series, e_series)
    print(f"Done (1L): {ic_key}")

# -------------------------
# Main
# -------------------------
if __name__ == "__main__":
    os.makedirs(ROOT_OUT, exist_ok=True)
    keys = list_ic_keys(IC_BUNDLE)
    print("ICs in bundle:", keys)
    for k in keys:
        run_ic(k)


ICs in bundle: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_04', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'mix_04', 'mix_05', 'test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_00', 'wave_01', 'wave_02', 'wave_03', 'wave_04', 'wave_05']

=== 1L IC: gauss_00 | nx=256, ny=128, dt=30.0s, nt=1200 ===
[save] gauss_00 step=0 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000000.npz
[    1] dM/M0=-6.563e-09  dE/E0=-1.336e-06  umax=  3.36
[  100] dM/M0=+1.280e-06  dE/E0=+1.054e-02  umax=  3.31
[save] gauss_00 step=200 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000200.npz
[  200] dM/M0=+5.043e-06  dE/E0=+3.638e-02  umax=  3.17
[  300] dM/M0=+1.121e-05  dE/E0=+6.995e-02  umax=  3.00
[save] gauss_00 step=400 -> /content/drive/MyDrive/AI_4DVAR/klein_ckpt_1L/gauss_00/klein_step_000400.npz
[  400] dM/M0=+1.970e-05  dE/E0=+1.074e-01  umax=  2.91
[  5

In [ ]:
from google.colab import drive
import os
def is_drive_mounted():
    return os.path.exists('/content/drive')
if not is_drive_mounted():
  drive.mount('/content/drive')

In [ ]:
# ============================================================
# train_cnn_emulator_1layer_multistep.py
# ------------------------------------------------------------
# Multi-step CNN emulator for 1-layer shallow-water forecasting
# on Klein/twisted beta-plane.
#
# Trains on grouped trajectories from train/val datasets:
#   X = [eta, uc, vc, f, y_norm] at macro time t_k
#   Y = [eta, uc, vc]             at macro time t_{k+1}
#
# Uses recursive rollout over multiple macro steps within each IC:
#   x_k -> x_{k+1} -> x_{k+2} -> ...
#
# Saves:
#   best_model.pt
#   norm_stats.npz
#   loss_history.csv
#   loss_curve.png
#   metrics.txt
#   rollout_plots/
#
# Colab usage:
# !python /content/train_cnn_emulator_1layer_multistep.py
# ============================================================

import os
import csv
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# -------------------------
# Paths
# -------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"
ROOT_DATA = f"{ROOT}/klein_ml_1L"
OUT_DIR   = f"{ROOT}/klein_ml_1L_ckpt_cnn_multistep"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(ROOT_DATA, "dataset_train_1L.npz")
VAL_PATH   = os.path.join(ROOT_DATA, "dataset_val_1L.npz")
TEST_PATH  = os.path.join(ROOT_DATA, "dataset_test_1L.npz")

# -------------------------
# Config
# -------------------------
class CFG:
    epochs = 200
    lr = 2e-3
    weight_decay = 1e-6
    print_every = 10

    width = 48
    grad_clip = 1.0

    # training rollout
    rollout_horizon = 4   # number of recursive steps in loss
    batch_traj = 4        # number of trajectory groups per optimizer step

    # loss weights
    lambda_eta = 1.0
    lambda_uv = 1.0
    lambda_mass = 1e-2

    # horizon weighting: later steps matter too
    rollout_gamma = 0.9

cfg = CFG()

# -------------------------
# Dataset loader
# -------------------------
def load_dataset(path):
    d = np.load(path, allow_pickle=True)
    return {
        "X": d["X"].astype(np.float32),
        "Y": d["Y"].astype(np.float32),
        "ic_key": d["ic_key"],
        "step0": d["step0"].astype(np.int32),
        "step1": d["step1"].astype(np.int32),
    }

train_data = load_dataset(TRAIN_PATH)
val_data   = load_dataset(VAL_PATH)
test_data  = load_dataset(TEST_PATH)

print("Train X:", train_data["X"].shape, "Y:", train_data["Y"].shape)
print("Val   X:", val_data["X"].shape,   "Y:", val_data["Y"].shape)
print("Test  X:", test_data["X"].shape,  "Y:", test_data["Y"].shape)

# -------------------------
# Normalization from TRAIN only
# -------------------------
def compute_channel_stats(X, Y):
    x_mean = X.mean(axis=(0,2,3), keepdims=True)
    x_std  = X.std(axis=(0,2,3), keepdims=True) + 1e-6
    y_mean = Y.mean(axis=(0,2,3), keepdims=True)
    y_std  = Y.std(axis=(0,2,3), keepdims=True) + 1e-6
    return x_mean.astype(np.float32), x_std.astype(np.float32), y_mean.astype(np.float32), y_std.astype(np.float32)

x_mean, x_std, y_mean, y_std = compute_channel_stats(train_data["X"], train_data["Y"])

np.savez_compressed(
    os.path.join(OUT_DIR, "norm_stats.npz"),
    x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std
)
print("Saved normalization stats.")

def normalize_X(X):
    return (X - x_mean) / x_std

def normalize_Y(Y):
    return (Y - y_mean) / y_std

def denormalize_Y(Yn):
    return Yn * y_std + y_mean

train_Xn = normalize_X(train_data["X"])
train_Yn = normalize_Y(train_data["Y"])
val_Xn   = normalize_X(val_data["X"])
val_Yn   = normalize_Y(val_data["Y"])
test_Xn  = normalize_X(test_data["X"])
test_Yn  = normalize_Y(test_data["Y"])

# torch copies of output normalization for mass penalty
y_mean_t = torch.tensor(y_mean, device=device)
y_std_t  = torch.tensor(y_std,  device=device)

# -------------------------
# Group samples by IC in time order
# -------------------------
def build_trajectories(data, Xn, Yn):
    """
    Returns list of trajectory dicts.
    Each trajectory contains ordered samples for one IC:
      Xn: (T, 5, ny, nx)
      Yn: (T, 3, ny, nx)
      step0, step1
    """
    groups = {}
    for i, ic in enumerate(data["ic_key"]):
        key = str(ic)
        groups.setdefault(key, []).append(i)

    trajs = []
    for key, idxs in groups.items():
        idxs = sorted(idxs, key=lambda j: data["step0"][j])
        trajs.append({
            "ic_key": key,
            "Xn": Xn[idxs].astype(np.float32),
            "Yn": Yn[idxs].astype(np.float32),
            "X_raw": data["X"][idxs].astype(np.float32),
            "Y_raw": data["Y"][idxs].astype(np.float32),
            "step0": data["step0"][idxs].astype(np.int32),
            "step1": data["step1"][idxs].astype(np.int32),
        })
    trajs = sorted(trajs, key=lambda d: d["ic_key"])
    return trajs

train_trajs = build_trajectories(train_data, train_Xn, train_Yn)
val_trajs   = build_trajectories(val_data,   val_Xn,   val_Yn)
test_trajs  = build_trajectories(test_data,  test_Xn,  test_Yn)

print("Train trajectories:", [t["ic_key"] for t in train_trajs])
print("Val trajectories  :", [t["ic_key"] for t in val_trajs])
print("Test trajectories :", [t["ic_key"] for t in test_trajs])

# -------------------------
# Model
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1),
            nn.GELU(),
        )
    def forward(self, x):
        return self.block(x)

class CNNEmulator(nn.Module):
    def __init__(self, cin=5, cout=3, width=48):
        super().__init__()
        self.in_proj = nn.Conv2d(cin, width, 3, padding=1)
        self.b1 = ConvBlock(width, width)
        self.b2 = ConvBlock(width, width)
        self.b3 = ConvBlock(width, width)
        self.out_proj = nn.Conv2d(width, cout, 1)

    def forward(self, x):
        z = self.in_proj(x)
        z = z + self.b1(z)
        z = z + self.b2(z)
        z = z + self.b3(z)
        return self.out_proj(z)

model = CNNEmulator(width=cfg.width).to(device)

# -------------------------
# Loss on one predicted step
# -------------------------
def one_step_loss(pred_n, targ_n):
    """
    pred_n, targ_n: normalized outputs, shape (B,3,ny,nx)
    """
    diff = pred_n - targ_n
    loss_eta = torch.mean(diff[:, 0:1]**2)
    loss_uv  = torch.mean(diff[:, 1:3]**2)

    pred_eta = pred_n[:, 0:1] * y_std_t[:, 0:1] + y_mean_t[:, 0:1]
    targ_eta = targ_n[:, 0:1] * y_std_t[:, 0:1] + y_mean_t[:, 0:1]
    mass_pen = torch.mean((pred_eta.mean(dim=(-2,-1)) - targ_eta.mean(dim=(-2,-1)))**2)

    loss = cfg.lambda_eta * loss_eta + cfg.lambda_uv * loss_uv + cfg.lambda_mass * mass_pen
    comps = {
        "eta": loss_eta.detach().item(),
        "uv": loss_uv.detach().item(),
        "mass": mass_pen.detach().item(),
    }
    return loss, comps

# -------------------------
# Recursive rollout loss on one trajectory
# -------------------------
def rollout_loss_for_trajectory(traj, training=True):
    """
    Rolls recursively for rollout_horizon steps over one IC trajectory.

    Important:
    - Start from actual X at first step in each window.
    - Replace only channels [eta,uc,vc] by model prediction.
    - Keep geometry channels [f,y_norm] from the actual next input.
    """
    Xn = torch.from_numpy(traj["Xn"]).to(device)   # (T,5,ny,nx)
    Yn = torch.from_numpy(traj["Yn"]).to(device)   # (T,3,ny,nx)

    T = Xn.shape[0]
    H = min(cfg.rollout_horizon, T)

    total_loss = torch.tensor(0.0, device=device)
    total_eta = 0.0
    total_uv = 0.0
    total_mass = 0.0
    n_terms = 0

    # windows starting at each valid start point
    for start in range(0, T):
        max_len = min(H, T - start)
        if max_len <= 0:
            continue

        x_cur = Xn[start:start+1].clone()  # (1,5,ny,nx)

        for lead in range(max_len):
            pred = model(x_cur)                # normalized (1,3,ny,nx)
            targ = Yn[start + lead:start + lead + 1]

            step_loss, comps = one_step_loss(pred, targ)
            w = cfg.rollout_gamma ** lead
            total_loss = total_loss + w * step_loss
            total_eta += w * comps["eta"]
            total_uv += w * comps["uv"]
            total_mass += w * comps["mass"]
            n_terms += 1

            # build next input recursively if not last available step
            if (start + lead + 1) < T:
                x_next = Xn[start + lead + 1:start + lead + 2].clone()
                x_next[:, 0:3] = pred
                x_cur = x_next

    if n_terms == 0:
        return torch.tensor(0.0, device=device), {"eta": 0.0, "uv": 0.0, "mass": 0.0}

    total_loss = total_loss / n_terms
    comps = {
        "eta": total_eta / n_terms,
        "uv": total_uv / n_terms,
        "mass": total_mass / n_terms,
    }
    return total_loss, comps

# -------------------------
# Optimizer
# -------------------------
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=15
)

# -------------------------
# Train/eval epoch
# -------------------------
def run_epoch(trajs, training):
    if training:
        model.train()
        order = np.random.permutation(len(trajs))
    else:
        model.eval()
        order = np.arange(len(trajs))

    loss_sum = 0.0
    eta_sum = 0.0
    uv_sum = 0.0
    mass_sum = 0.0
    n = 0

    # process trajectory groups; optionally accumulate small batches
    batch_losses = []
    batch_eta = 0.0
    batch_uv = 0.0
    batch_mass = 0.0
    batch_count = 0

    for idx in order:
        traj = trajs[idx]

        with torch.set_grad_enabled(training):
            loss, comps = rollout_loss_for_trajectory(traj, training=training)

        if training:
            batch_losses.append(loss)
            batch_eta += comps["eta"]
            batch_uv += comps["uv"]
            batch_mass += comps["mass"]
            batch_count += 1

            if batch_count == cfg.batch_traj:
                batch_loss = torch.stack(batch_losses).mean()

                if torch.isfinite(batch_loss):
                    optimizer.zero_grad(set_to_none=True)
                    batch_loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                    optimizer.step()

                loss_sum += batch_loss.detach().item() * batch_count
                eta_sum += batch_eta
                uv_sum += batch_uv
                mass_sum += batch_mass
                n += batch_count

                batch_losses = []
                batch_eta = batch_uv = batch_mass = 0.0
                batch_count = 0

        else:
            loss_sum += loss.detach().item()
            eta_sum += comps["eta"]
            uv_sum += comps["uv"]
            mass_sum += comps["mass"]
            n += 1

    # flush last partial batch
    if training and batch_count > 0:
        batch_loss = torch.stack(batch_losses).mean()
        if torch.isfinite(batch_loss):
            optimizer.zero_grad(set_to_none=True)
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

        loss_sum += batch_loss.detach().item() * batch_count
        eta_sum += batch_eta
        uv_sum += batch_uv
        mass_sum += batch_mass
        n += batch_count

    return {
        "loss": loss_sum / max(n, 1),
        "eta": eta_sum / max(n, 1),
        "uv": uv_sum / max(n, 1),
        "mass": mass_sum / max(n, 1),
    }

# -------------------------
# Train loop
# -------------------------
history = []
best_val = float("inf")
best_path = os.path.join(OUT_DIR, "best_model.pt")

for ep in range(cfg.epochs):
    tr = run_epoch(train_trajs, training=True)
    va = run_epoch(val_trajs, training=False)

    scheduler.step(va["loss"])

    history.append({
        "epoch": ep,
        "train": tr["loss"],
        "val": va["loss"],
        "train_eta": tr["eta"],
        "train_uv": tr["uv"],
        "train_mass": tr["mass"],
        "val_eta": va["eta"],
        "val_uv": va["uv"],
        "val_mass": va["mass"],
        "lr": optimizer.param_groups[0]["lr"],
    })

    if va["loss"] < best_val:
        best_val = va["loss"]
        torch.save(model.state_dict(), best_path)

    if (ep % cfg.print_every) == 0 or ep == cfg.epochs - 1:
        print(
            f"[ep {ep:03d}] "
            f"train={tr['loss']:.4e} val={va['loss']:.4e} "
            f"(eta tr/va={tr['eta']:.3e}/{va['eta']:.3e}, "
            f"uv tr/va={tr['uv']:.3e}/{va['uv']:.3e}, "
            f"mass tr/va={tr['mass']:.3e}/{va['mass']:.3e}) "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

print("Best val loss:", best_val)
print("Saved:", best_path)

# -------------------------
# Save history
# -------------------------
hist_csv = os.path.join(OUT_DIR, "loss_history.csv")
with open(hist_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    w.writeheader()
    w.writerows(history)
print("Saved:", hist_csv)

plt.figure(figsize=(7,4))
plt.plot([h["epoch"] for h in history], [h["train"] for h in history], label="train")
plt.plot([h["epoch"] for h in history], [h["val"] for h in history], label="val")
plt.xlabel("epoch")
plt.ylabel("multistep loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("CNN multistep emulator loss")
plt.tight_layout()
loss_png = os.path.join(OUT_DIR, "loss_curve.png")
plt.savefig(loss_png, dpi=140)
plt.close()
print("Saved:", loss_png)

# -------------------------
# Reload best and evaluate rollout-like per IC
# -------------------------
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

def rollout_one_ic(traj):
    """
    Recursive rollout over full trajectory length for one IC.
    Returns denormalized predictions and denormalized truths.
    """
    Xn = traj["Xn"]
    Yn = traj["Yn"]

    x_cur = Xn[0:1].copy()
    preds_n = []

    with torch.no_grad():
        for k in range(len(Xn)):
            xb = torch.from_numpy(x_cur).to(device)
            yp_n = model(xb).cpu().numpy()      # (1,3,ny,nx)
            preds_n.append(yp_n[0])

            if k < len(Xn) - 1:
                x_next = Xn[k+1:k+2].copy()
                x_next[:, 0:3] = yp_n
                x_cur = x_next

    preds_n = np.stack(preds_n, axis=0)
    truths_n = Yn[:len(preds_n)]

    preds = denormalize_Y(preds_n)
    truths = denormalize_Y(truths_n)
    return preds, truths

def channel_rmse(pred, true):
    return {
        "eta": float(np.sqrt(np.mean((pred[:,0] - true[:,0])**2))),
        "uc":  float(np.sqrt(np.mean((pred[:,1] - true[:,1])**2))),
        "vc":  float(np.sqrt(np.mean((pred[:,2] - true[:,2])**2))),
    }

metrics_txt = os.path.join(OUT_DIR, "metrics.txt")
roll_dir = os.path.join(OUT_DIR, "rollout_plots")
os.makedirs(roll_dir, exist_ok=True)

with open(metrics_txt, "w") as f:
    f.write("Validation rollout-like RMSE by IC\n")
    val_rmses = []
    for traj in val_trajs:
        pred, true = rollout_one_ic(traj)
        met = channel_rmse(pred, true)
        val_rmses.append([met["eta"], met["uc"], met["vc"]])

        # eta RMSE by lead
        eta_rmse = [math.sqrt(np.mean((pred[k,0] - true[k,0])**2)) for k in range(len(pred))]
        plt.figure(figsize=(6,4))
        plt.plot(range(1, len(eta_rmse)+1), eta_rmse, "-o")
        plt.xlabel("macro lead step")
        plt.ylabel("RMSE eta")
        plt.grid(True, alpha=0.3)
        plt.title(f"VAL rollout-like: {traj['ic_key']}")
        plt.tight_layout()
        out = os.path.join(roll_dir, f"val_{traj['ic_key']}_eta_rmse.png")
        plt.savefig(out, dpi=140)
        plt.close()

        f.write(f"{traj['ic_key']}: {met}\n")

    val_rmses = np.array(val_rmses)
    val_mean = {
        "eta": float(val_rmses[:,0].mean()),
        "uc":  float(val_rmses[:,1].mean()),
        "vc":  float(val_rmses[:,2].mean()),
    }
    f.write(f"\nValidation mean rollout-like RMSE: {val_mean}\n")

    f.write("\nTest rollout-like RMSE by IC\n")
    test_rmses = []
    for traj in test_trajs:
        pred, true = rollout_one_ic(traj)
        met = channel_rmse(pred, true)
        test_rmses.append([met["eta"], met["uc"], met["vc"]])

        eta_rmse = [math.sqrt(np.mean((pred[k,0] - true[k,0])**2)) for k in range(len(pred))]
        plt.figure(figsize=(6,4))
        plt.plot(range(1, len(eta_rmse)+1), eta_rmse, "-o")
        plt.xlabel("macro lead step")
        plt.ylabel("RMSE eta")
        plt.grid(True, alpha=0.3)
        plt.title(f"TEST rollout-like: {traj['ic_key']}")
        plt.tight_layout()
        out = os.path.join(roll_dir, f"test_{traj['ic_key']}_eta_rmse.png")
        plt.savefig(out, dpi=140)
        plt.close()

        f.write(f"{traj['ic_key']}: {met}\n")

    test_rmses = np.array(test_rmses)
    test_mean = {
        "eta": float(test_rmses[:,0].mean()),
        "uc":  float(test_rmses[:,1].mean()),
        "vc":  float(test_rmses[:,2].mean()),
    }
    f.write(f"\nTest mean rollout-like RMSE: {test_mean}\n")

print("Saved:", metrics_txt)
print("Saved rollout plots in:", roll_dir)
print("Done.")

Train X: (114, 5, 128, 256) Y: (114, 3, 128, 256)
Val   X: (30, 5, 128, 256) Y: (30, 3, 128, 256)
Test  X: (24, 5, 128, 256) Y: (24, 3, 128, 256)
Saved normalization stats.
Train trajectories: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_01', 'wave_02', 'wave_04', 'wave_05']
Val trajectories  : ['gauss_04', 'mix_04', 'mix_05', 'wave_00', 'wave_03']
Test trajectories : ['test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01']
[ep 000] train=2.0028e+00 val=2.9241e+00 (eta tr/va=9.441e-01/1.186e+00, uv tr/va=8.040e-01/7.225e-01, mass tr/va=2.546e+01/1.016e+02) lr=2.00e-03
[ep 010] train=9.1686e-01 val=1.4897e+00 (eta tr/va=5.557e-01/9.413e-01, uv tr/va=3.419e-01/5.133e-01, mass tr/va=1.924e+00/3.507e+00) lr=2.00e-03
[ep 020] train=8.3016e-01 val=1.4348e+00 (eta tr/va=5.324e-01/9.164e-01, uv tr/va=2.783e-01/4.562e-01, mass tr/va=1.941e+00/6.226e+00) lr=2.00e-

# Evaluation of multistep model

In [ ]:
# ==============================================
# eval_cnn_emulator_1layer.py
# Corrected version for datasets with:
# X, Y, ic_key, step0, step1
# ==============================================

import os
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = "/content/drive/MyDrive/AI_4DVAR"

DATA_TEST = f"{ROOT}/klein_ml_1L/dataset_test_1L.npz"
DATA_VAL  = f"{ROOT}/klein_ml_1L/dataset_val_1L.npz"

MODEL_PATH = f"{ROOT}/klein_ml_1L_ckpt_cnn_multistep/best_model.pt"
OUT_DIR    = f"{ROOT}/klein_ml_1L_ckpt_cnn_multistep"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1),
            nn.GELU(),
        )
    def forward(self, x):
        return self.block(x)

class CNNEmulator(nn.Module):
    def __init__(self, cin=5, cout=3, width=48):
        super().__init__()
        self.in_proj = nn.Conv2d(cin, width, 3, padding=1)
        self.b1 = ConvBlock(width, width)
        self.b2 = ConvBlock(width, width)
        self.b3 = ConvBlock(width, width)
        self.out_proj = nn.Conv2d(width, cout, 1)

    def forward(self, x):
        z = self.in_proj(x)
        z = z + self.b1(z)
        z = z + self.b2(z)
        z = z + self.b3(z)
        return self.out_proj(z)

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def load_npz(path):
    d = np.load(path, allow_pickle=True)
    return d["X"], d["Y"], d["ic_key"], d["step0"], d["step1"]

X_val, Y_val, ic_val, step0_val, step1_val = load_npz(DATA_VAL)
X_test, Y_test, ic_test, step0_test, step1_test = load_npz(DATA_TEST)

print("Val:", X_val.shape, Y_val.shape)
print("Test:", X_test.shape, Y_test.shape)

model = CNNEmulator().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

def evaluate_one_step(X, Y):
    with torch.no_grad():
        x = torch.tensor(X, dtype=torch.float32).to(DEVICE)
        y_true = torch.tensor(Y, dtype=torch.float32).to(DEVICE)
        y_pred = model(x)
    y_pred = y_pred.cpu().numpy()
    y_true = y_true.cpu().numpy()

    out = {
        "eta": rmse(y_pred[:, 0], y_true[:, 0]),
        "uc":  rmse(y_pred[:, 1], y_true[:, 1]),
        "vc":  rmse(y_pred[:, 2], y_true[:, 2]),
    }
    return out, y_pred

rmse_val, pred_val = evaluate_one_step(X_val, Y_val)
rmse_test, pred_test = evaluate_one_step(X_test, Y_test)

print("\nValidation RMSE:", rmse_val)
print("Test RMSE:", rmse_test)

with open(f"{OUT_DIR}/metrics.txt", "w") as f:
    f.write(f"Validation RMSE: {rmse_val}\n")
    f.write(f"Test RMSE: {rmse_test}\n")

PLOT_DIR = os.path.join(OUT_DIR, "predictions")
os.makedirs(PLOT_DIR, exist_ok=True)

def plot_sample(y_true, y_pred, title, out_png):
    fields = ["eta", "uc", "vc"]
    plt.figure(figsize=(12, 6))
    for i in range(3):
        plt.subplot(3, 3, i*3+1)
        plt.imshow(y_true[i], origin="lower")
        plt.title(f"{fields[i]} truth")
        plt.colorbar()

        plt.subplot(3, 3, i*3+2)
        plt.imshow(y_pred[i], origin="lower")
        plt.title(f"{fields[i]} pred")
        plt.colorbar()

        plt.subplot(3, 3, i*3+3)
        plt.imshow(y_pred[i] - y_true[i], origin="lower")
        plt.title(f"{fields[i]} error")
        plt.colorbar()
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=120)
    plt.close()

for i in range(min(5, len(X_test))):
    title = f"test {ic_test[i]} {step0_test[i]}->{step1_test[i]}"
    plot_sample(Y_test[i], pred_test[i], title, f"{PLOT_DIR}/test_{i}.png")

ROLL_DIR = os.path.join(OUT_DIR, "rollout_like")
os.makedirs(ROLL_DIR, exist_ok=True)

def normalize_X(X, x_mean, x_std):
    return (X - x_mean) / x_std

# load norm stats
norm = np.load(f"{OUT_DIR}/norm_stats.npz")
x_mean = norm["x_mean"].astype(np.float32)
x_std  = norm["x_std"].astype(np.float32)
y_mean = norm["y_mean"].astype(np.float32)
y_std  = norm["y_std"].astype(np.float32)

def denormalize_Y(Yn):
    return Yn * y_std + y_mean

def group_by_ic(ic_keys):
    out = {}
    for i, ic in enumerate(ic_keys):
        out.setdefault(str(ic), []).append(i)
    return out

def rollout_one_ic(X_raw_ic):
    x_cur_raw = X_raw_ic[0:1].copy()
    preds = []
    with torch.no_grad():
        for _ in range(len(X_raw_ic)):
            x_cur = normalize_X(x_cur_raw, x_mean, x_std)
            xb = torch.tensor(x_cur, dtype=torch.float32).to(DEVICE)
            yp_n = model(xb).cpu().numpy()
            yp = denormalize_Y(yp_n)[0]
            preds.append(yp)

            # next input: replace eta,uc,vc with predicted values
            if len(preds) < len(X_raw_ic):
                x_next_raw = X_raw_ic[len(preds):len(preds)+1].copy()
                x_next_raw[0, 0:3] = yp
                x_cur_raw = x_next_raw
    return np.array(preds)

def evaluate_rollout(X, Y, ic_keys, step0, tag):
    groups = group_by_ic(ic_keys)
    results = []

    for ic in sorted(groups.keys()):
        idx = sorted(groups[ic], key=lambda j: step0[j])
        X_ic = X[idx]
        Y_ic = Y[idx]

        pred = rollout_one_ic(X_ic)
        truth = Y_ic[:len(pred)]

        eta_rmse = [rmse(pred[k, 0], truth[k, 0]) for k in range(len(pred))]
        uc_rmse  = [rmse(pred[k, 1], truth[k, 1]) for k in range(len(pred))]
        vc_rmse  = [rmse(pred[k, 2], truth[k, 2]) for k in range(len(pred))]

        plt.figure(figsize=(7,4))
        plt.plot(eta_rmse, "-o", label="eta")
        plt.plot(uc_rmse, "-s", label="uc")
        plt.plot(vc_rmse, "-^", label="vc")
        plt.xlabel("macro lead step")
        plt.ylabel("RMSE")
        plt.title(f"{tag} rollout-like: {ic}")
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"{ROLL_DIR}/rollout_{tag}_{ic}.png", dpi=120)
        plt.close()

        results.append(
            f"{tag:5s} {ic:16s}  final_eta_rmse={eta_rmse[-1]:.6e}  "
            f"final_uc_rmse={uc_rmse[-1]:.6e}  final_vc_rmse={vc_rmse[-1]:.6e}"
        )

    return results

roll_val = evaluate_rollout(X_val, Y_val, ic_val, step0_val, "val")
roll_test = evaluate_rollout(X_test, Y_test, ic_test, step0_test, "test")

with open(f"{OUT_DIR}/rollout_like_summary.txt", "w") as f:
    f.write("VAL:\n")
    for line in roll_val:
        f.write(line + "\n")
    f.write("\nTEST:\n")
    for line in roll_test:
        f.write(line + "\n")

print("Saved rollout_like_summary.txt")
print("Done.")

Val: (30, 5, 128, 256) (30, 3, 128, 256)
Test: (24, 5, 128, 256) (24, 3, 128, 256)

Validation RMSE: {'eta': 27.851566314697266, 'uc': 6.167268753051758, 'vc': 15.083577156066895}
Test RMSE: {'eta': 86.33952331542969, 'uc': 18.39808464050293, 'vc': 37.04530334472656}
Saved rollout_like_summary.txt
Done.


# Repeat training of multistep version with minimal physics

## train_cnn_emulator_1layer_multistep_physcis.py

In [ ]:
# ============================================================
# train_cnn_emulator_1layer_multistep_physics.py
# ------------------------------------------------------------
# Multistep CNN emulator for 1-layer shallow-water forecasting
# with added centered-grid continuity residual loss.
#
# Input channels:
#   [eta, uc, vc, f, y_norm]
#
# Target channels:
#   [eta, uc, vc] at next saved macro time
#
# Physics penalty:
#   r = (eta_pred - eta_in)/dt_macro + H*(dux + dvy)
#
# Saves:
#   best_model.pt
#   norm_stats.npz
#   loss_history.csv
#   loss_curve.png
#   metrics.txt
#   rollout_plots/
# ============================================================

import os
import csv
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# -------------------------
# Paths
# -------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"
ROOT_DATA = f"{ROOT}/klein_ml_1L"
OUT_DIR   = f"{ROOT}/klein_ml_1L_ckpt_cnn_multistep_physics"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(ROOT_DATA, "dataset_train_1L.npz")
VAL_PATH   = os.path.join(ROOT_DATA, "dataset_val_1L.npz")
TEST_PATH  = os.path.join(ROOT_DATA, "dataset_test_1L.npz")

# -------------------------
# Config
# -------------------------
class CFG:
    epochs = 200
    lr = 2e-3
    weight_decay = 1e-6
    print_every = 10

    width = 48
    grad_clip = 1.0

    rollout_horizon = 4
    batch_traj = 4
    rollout_gamma = 0.9

    # data loss weights
    lambda_eta = 1.0
    lambda_uv = 1.0
    lambda_mass = 1e-2

    # physics loss weight
    lambda_phys = 1e-4

cfg = CFG()

# -------------------------
# Physical constants
# -------------------------
H = 1000.0
dt_macro = 200.0 * 30.0  # 6000 s from your saved 0->200 steps

# -------------------------
# Dataset loader
# -------------------------
def load_dataset(path):
    d = np.load(path, allow_pickle=True)
    return {
        "X": d["X"].astype(np.float32),
        "Y": d["Y"].astype(np.float32),
        "ic_key": d["ic_key"],
        "step0": d["step0"].astype(np.int32),
        "step1": d["step1"].astype(np.int32),
    }

train_data = load_dataset(TRAIN_PATH)
val_data   = load_dataset(VAL_PATH)
test_data  = load_dataset(TEST_PATH)

print("Train X:", train_data["X"].shape, "Y:", train_data["Y"].shape)
print("Val   X:", val_data["X"].shape,   "Y:", val_data["Y"].shape)
print("Test  X:", test_data["X"].shape,  "Y:", test_data["Y"].shape)

# -------------------------
# Grid spacing from raw data
# -------------------------
# f and y_norm are in dataset, but dx/dy are not. Use known grid.
nx = train_data["X"].shape[-1]
ny = train_data["X"].shape[-2]
Lx = 2.0e7
Ly = 8.0e6
dx = Lx / nx
dy = Ly / ny

# -------------------------
# Normalization from TRAIN only
# -------------------------
def compute_channel_stats(X, Y):
    x_mean = X.mean(axis=(0,2,3), keepdims=True)
    x_std  = X.std(axis=(0,2,3), keepdims=True) + 1e-6
    y_mean = Y.mean(axis=(0,2,3), keepdims=True)
    y_std  = Y.std(axis=(0,2,3), keepdims=True) + 1e-6
    return x_mean.astype(np.float32), x_std.astype(np.float32), y_mean.astype(np.float32), y_std.astype(np.float32)

x_mean, x_std, y_mean, y_std = compute_channel_stats(train_data["X"], train_data["Y"])

np.savez_compressed(
    os.path.join(OUT_DIR, "norm_stats.npz"),
    x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std
)
print("Saved normalization stats.")

def normalize_X(X):
    return (X - x_mean) / x_std

def normalize_Y(Y):
    return (Y - y_mean) / y_std

def denormalize_Y(Yn):
    return Yn * y_std + y_mean

train_Xn = normalize_X(train_data["X"])
train_Yn = normalize_Y(train_data["Y"])
val_Xn   = normalize_X(val_data["X"])
val_Yn   = normalize_Y(val_data["Y"])
test_Xn  = normalize_X(test_data["X"])
test_Yn  = normalize_Y(test_data["Y"])

# torch copies for denorm inside loss
y_mean_t = torch.tensor(y_mean, device=device)
y_std_t  = torch.tensor(y_std,  device=device)

# -------------------------
# Group samples by IC in time order
# -------------------------
def build_trajectories(data, Xn, Yn):
    groups = {}
    for i, ic in enumerate(data["ic_key"]):
        key = str(ic)
        groups.setdefault(key, []).append(i)

    trajs = []
    for key, idxs in groups.items():
        idxs = sorted(idxs, key=lambda j: data["step0"][j])
        trajs.append({
            "ic_key": key,
            "Xn": Xn[idxs].astype(np.float32),
            "Yn": Yn[idxs].astype(np.float32),
            "X_raw": data["X"][idxs].astype(np.float32),
            "Y_raw": data["Y"][idxs].astype(np.float32),
            "step0": data["step0"][idxs].astype(np.int32),
            "step1": data["step1"][idxs].astype(np.int32),
        })
    trajs = sorted(trajs, key=lambda d: d["ic_key"])
    return trajs

train_trajs = build_trajectories(train_data, train_Xn, train_Yn)
val_trajs   = build_trajectories(val_data,   val_Xn,   val_Yn)
test_trajs  = build_trajectories(test_data,  test_Xn,  test_Yn)

print("Train trajectories:", [t["ic_key"] for t in train_trajs])
print("Val trajectories  :", [t["ic_key"] for t in val_trajs])
print("Test trajectories :", [t["ic_key"] for t in test_trajs])

# -------------------------
# Model
# -------------------------
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(cout, cout, 3, padding=1),
            nn.GELU(),
        )
    def forward(self, x):
        return self.block(x)

class CNNEmulator(nn.Module):
    def __init__(self, cin=5, cout=3, width=48):
        super().__init__()
        self.in_proj = nn.Conv2d(cin, width, 3, padding=1)
        self.b1 = ConvBlock(width, width)
        self.b2 = ConvBlock(width, width)
        self.b3 = ConvBlock(width, width)
        self.out_proj = nn.Conv2d(width, cout, 1)

    def forward(self, x):
        z = self.in_proj(x)
        z = z + self.b1(z)
        z = z + self.b2(z)
        z = z + self.b3(z)
        return self.out_proj(z)

model = CNNEmulator(width=cfg.width).to(device)

# -------------------------
# Centered derivatives
# -------------------------
def ddx_center(a):
    return (torch.roll(a, shifts=-1, dims=-1) - torch.roll(a, shifts=1, dims=-1)) / (2.0 * dx)

def ddy_center(a):
    a_up = torch.cat([a[:, :, 0:1, :], a[:, :, :-1, :]], dim=-2)
    a_dn = torch.cat([a[:, :, 1:, :], a[:, :, -1:, :]], dim=-2)
    return (a_dn - a_up) / (2.0 * dy)

# -------------------------
# Loss on one predicted step
# -------------------------
def one_step_loss(pred_n, targ_n, x_in_n):
    """
    pred_n: normalized predicted output, shape (B,3,ny,nx)
    targ_n: normalized target output,    shape (B,3,ny,nx)
    x_in_n: normalized input state,      shape (B,5,ny,nx)
    """
    # data loss in normalized space
    diff = pred_n - targ_n
    loss_eta = torch.mean(diff[:, 0:1]**2)
    loss_uv  = torch.mean(diff[:, 1:3]**2)

    # denormalize for physics/mass
    pred = pred_n * y_std_t + y_mean_t       # (B,3,ny,nx)
    targ = targ_n * y_std_t + y_mean_t
    x_in_phys = x_in_n[:, 0:3] * torch.tensor(x_std[:, 0:3], device=device) + torch.tensor(x_mean[:, 0:3], device=device)

    pred_eta = pred[:, 0:1]
    pred_uc  = pred[:, 1:2]
    pred_vc  = pred[:, 2:3]

    in_eta = x_in_phys[:, 0:1]

    # mass penalty on eta mean
    targ_eta = targ[:, 0:1]
    mass_pen = torch.mean((pred_eta.mean(dim=(-2,-1)) - targ_eta.mean(dim=(-2,-1)))**2)

    # linearized continuity residual:
    # (eta_pred - eta_in)/dt + H*(dux + dvy)
    dux = ddx_center(pred_uc)
    dvy = ddy_center(pred_vc)
    resid = (pred_eta - in_eta) / dt_macro + H * (dux + dvy)
    phys_pen = torch.mean(resid**2)

    loss = (
        cfg.lambda_eta * loss_eta +
        cfg.lambda_uv  * loss_uv +
        cfg.lambda_mass * mass_pen +
        cfg.lambda_phys * phys_pen
    )

    comps = {
        "eta": loss_eta.detach().item(),
        "uv": loss_uv.detach().item(),
        "mass": mass_pen.detach().item(),
        "phys": phys_pen.detach().item(),
    }
    return loss, comps

# -------------------------
# Recursive rollout loss on one trajectory
# -------------------------
def rollout_loss_for_trajectory(traj):
    Xn = torch.from_numpy(traj["Xn"]).to(device)   # (T,5,ny,nx)
    Yn = torch.from_numpy(traj["Yn"]).to(device)   # (T,3,ny,nx)

    T = Xn.shape[0]
    Hh = min(cfg.rollout_horizon, T)

    total_loss = torch.tensor(0.0, device=device)
    total_eta = 0.0
    total_uv = 0.0
    total_mass = 0.0
    total_phys = 0.0
    n_terms = 0

    for start in range(T):
        max_len = min(Hh, T - start)
        if max_len <= 0:
            continue

        x_cur = Xn[start:start+1].clone()

        for lead in range(max_len):
            pred = model(x_cur)
            targ = Yn[start + lead:start + lead + 1]

            step_loss, comps = one_step_loss(pred, targ, x_cur)
            w = cfg.rollout_gamma ** lead

            total_loss = total_loss + w * step_loss
            total_eta += w * comps["eta"]
            total_uv += w * comps["uv"]
            total_mass += w * comps["mass"]
            total_phys += w * comps["phys"]
            n_terms += 1

            if (start + lead + 1) < T:
                x_next = Xn[start + lead + 1:start + lead + 2].clone()
                x_next[:, 0:3] = pred
                x_cur = x_next

    if n_terms == 0:
        return torch.tensor(0.0, device=device), {"eta": 0.0, "uv": 0.0, "mass": 0.0, "phys": 0.0}

    total_loss = total_loss / n_terms
    comps = {
        "eta": total_eta / n_terms,
        "uv": total_uv / n_terms,
        "mass": total_mass / n_terms,
        "phys": total_phys / n_terms,
    }
    return total_loss, comps

# -------------------------
# Optimizer
# -------------------------
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=15
)

# -------------------------
# Train/eval epoch
# -------------------------
def run_epoch(trajs, training):
    if training:
        model.train()
        order = np.random.permutation(len(trajs))
    else:
        model.eval()
        order = np.arange(len(trajs))

    loss_sum = 0.0
    eta_sum = 0.0
    uv_sum = 0.0
    mass_sum = 0.0
    phys_sum = 0.0
    n = 0

    batch_losses = []
    batch_eta = 0.0
    batch_uv = 0.0
    batch_mass = 0.0
    batch_phys = 0.0
    batch_count = 0

    for idx in order:
        traj = trajs[idx]

        with torch.set_grad_enabled(training):
            loss, comps = rollout_loss_for_trajectory(traj)

        if training:
            batch_losses.append(loss)
            batch_eta += comps["eta"]
            batch_uv += comps["uv"]
            batch_mass += comps["mass"]
            batch_phys += comps["phys"]
            batch_count += 1

            if batch_count == cfg.batch_traj:
                batch_loss = torch.stack(batch_losses).mean()

                if torch.isfinite(batch_loss):
                    optimizer.zero_grad(set_to_none=True)
                    batch_loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                    optimizer.step()

                loss_sum += batch_loss.detach().item() * batch_count
                eta_sum += batch_eta
                uv_sum += batch_uv
                mass_sum += batch_mass
                phys_sum += batch_phys
                n += batch_count

                batch_losses = []
                batch_eta = batch_uv = batch_mass = batch_phys = 0.0
                batch_count = 0
        else:
            loss_sum += loss.detach().item()
            eta_sum += comps["eta"]
            uv_sum += comps["uv"]
            mass_sum += comps["mass"]
            phys_sum += comps["phys"]
            n += 1

    if training and batch_count > 0:
        batch_loss = torch.stack(batch_losses).mean()
        if torch.isfinite(batch_loss):
            optimizer.zero_grad(set_to_none=True)
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

        loss_sum += batch_loss.detach().item() * batch_count
        eta_sum += batch_eta
        uv_sum += batch_uv
        mass_sum += batch_mass
        phys_sum += batch_phys
        n += batch_count

    return {
        "loss": loss_sum / max(n, 1),
        "eta": eta_sum / max(n, 1),
        "uv": uv_sum / max(n, 1),
        "mass": mass_sum / max(n, 1),
        "phys": phys_sum / max(n, 1),
    }

# -------------------------
# Train loop
# -------------------------
history = []
best_val = float("inf")
best_path = os.path.join(OUT_DIR, "best_model.pt")

for ep in range(cfg.epochs):
    tr = run_epoch(train_trajs, training=True)
    va = run_epoch(val_trajs, training=False)

    scheduler.step(va["loss"])

    history.append({
        "epoch": ep,
        "train": tr["loss"],
        "val": va["loss"],
        "train_eta": tr["eta"],
        "train_uv": tr["uv"],
        "train_mass": tr["mass"],
        "train_phys": tr["phys"],
        "val_eta": va["eta"],
        "val_uv": va["uv"],
        "val_mass": va["mass"],
        "val_phys": va["phys"],
        "lr": optimizer.param_groups[0]["lr"],
    })

    if va["loss"] < best_val:
        best_val = va["loss"]
        torch.save(model.state_dict(), best_path)

    if (ep % cfg.print_every) == 0 or ep == cfg.epochs - 1:
        print(
            f"[ep {ep:03d}] "
            f"train={tr['loss']:.4e} val={va['loss']:.4e} "
            f"(eta tr/va={tr['eta']:.3e}/{va['eta']:.3e}, "
            f"uv tr/va={tr['uv']:.3e}/{va['uv']:.3e}, "
            f"mass tr/va={tr['mass']:.3e}/{va['mass']:.3e}, "
            f"phys tr/va={tr['phys']:.3e}/{va['phys']:.3e}) "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

print("Best val loss:", best_val)
print("Saved:", best_path)

# -------------------------
# Save history
# -------------------------
hist_csv = os.path.join(OUT_DIR, "loss_history.csv")
with open(hist_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    w.writeheader()
    w.writerows(history)
print("Saved:", hist_csv)

plt.figure(figsize=(7,4))
plt.plot([h["epoch"] for h in history], [h["train"] for h in history], label="train")
plt.plot([h["epoch"] for h in history], [h["val"] for h in history], label="val")
plt.xlabel("epoch")
plt.ylabel("multistep+physics loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("CNN multistep+physics emulator loss")
plt.tight_layout()
loss_png = os.path.join(OUT_DIR, "loss_curve.png")
plt.savefig(loss_png, dpi=140)
plt.close()
print("Saved:", loss_png)

# -------------------------
# Reload best and evaluate rollout-like per IC
# -------------------------
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

def rollout_one_ic(traj):
    Xn = traj["Xn"]
    Yn = traj["Yn"]

    x_cur = Xn[0:1].copy()
    preds_n = []

    with torch.no_grad():
        for k in range(len(Xn)):
            xb = torch.from_numpy(x_cur).to(device)
            yp_n = model(xb).cpu().numpy()
            preds_n.append(yp_n[0])

            if k < len(Xn) - 1:
                x_next = Xn[k+1:k+2].copy()
                x_next[:, 0:3] = yp_n
                x_cur = x_next

    preds_n = np.stack(preds_n, axis=0)
    truths_n = Yn[:len(preds_n)]

    preds = denormalize_Y(preds_n)
    truths = denormalize_Y(truths_n)
    return preds, truths

def channel_rmse(pred, true):
    return {
        "eta": float(np.sqrt(np.mean((pred[:,0] - true[:,0])**2))),
        "uc":  float(np.sqrt(np.mean((pred[:,1] - true[:,1])**2))),
        "vc":  float(np.sqrt(np.mean((pred[:,2] - true[:,2])**2))),
    }

metrics_txt = os.path.join(OUT_DIR, "metrics.txt")
roll_dir = os.path.join(OUT_DIR, "rollout_plots")
os.makedirs(roll_dir, exist_ok=True)

with open(metrics_txt, "w") as f:
    f.write("Validation rollout-like RMSE by IC\n")
    val_rmses = []
    for traj in val_trajs:
        pred, true = rollout_one_ic(traj)
        met = channel_rmse(pred, true)
        val_rmses.append([met["eta"], met["uc"], met["vc"]])

        eta_rmse = [math.sqrt(np.mean((pred[k,0] - true[k,0])**2)) for k in range(len(pred))]
        plt.figure(figsize=(6,4))
        plt.plot(range(1, len(eta_rmse)+1), eta_rmse, "-o")
        plt.xlabel("macro lead step")
        plt.ylabel("RMSE eta")
        plt.grid(True, alpha=0.3)
        plt.title(f"VAL rollout-like: {traj['ic_key']}")
        plt.tight_layout()
        out = os.path.join(roll_dir, f"val_{traj['ic_key']}_eta_rmse.png")
        plt.savefig(out, dpi=140)
        plt.close()

        f.write(f"{traj['ic_key']}: {met}\n")

    val_rmses = np.array(val_rmses)
    val_mean = {
        "eta": float(val_rmses[:,0].mean()),
        "uc":  float(val_rmses[:,1].mean()),
        "vc":  float(val_rmses[:,2].mean()),
    }
    f.write(f"\nValidation mean rollout-like RMSE: {val_mean}\n")

    f.write("\nTest rollout-like RMSE by IC\n")
    test_rmses = []
    for traj in test_trajs:
        pred, true = rollout_one_ic(traj)
        met = channel_rmse(pred, true)
        test_rmses.append([met["eta"], met["uc"], met["vc"]])

        eta_rmse = [math.sqrt(np.mean((pred[k,0] - true[k,0])**2)) for k in range(len(pred))]
        plt.figure(figsize=(6,4))
        plt.plot(range(1, len(eta_rmse)+1), eta_rmse, "-o")
        plt.xlabel("macro lead step")
        plt.ylabel("RMSE eta")
        plt.grid(True, alpha=0.3)
        plt.title(f"TEST rollout-like: {traj['ic_key']}")
        plt.tight_layout()
        out = os.path.join(roll_dir, f"test_{traj['ic_key']}_eta_rmse.png")
        plt.savefig(out, dpi=140)
        plt.close()

        f.write(f"{traj['ic_key']}: {met}\n")

    test_rmses = np.array(test_rmses)
    test_mean = {
        "eta": float(test_rmses[:,0].mean()),
        "uc":  float(test_rmses[:,1].mean()),
        "vc":  float(test_rmses[:,2].mean()),
    }
    f.write(f"\nTest mean rollout-like RMSE: {test_mean}\n")

print("Saved:", metrics_txt)
print("Saved rollout plots in:", roll_dir)
print("Done.")

Train X: (114, 5, 128, 256) Y: (114, 3, 128, 256)
Val   X: (30, 5, 128, 256) Y: (30, 3, 128, 256)
Test  X: (24, 5, 128, 256) Y: (24, 3, 128, 256)
Saved normalization stats.
Train trajectories: ['gauss_00', 'gauss_01', 'gauss_02', 'gauss_03', 'gauss_05', 'mix_00', 'mix_01', 'mix_02', 'mix_03', 'vort_00', 'vort_01', 'vort_02', 'vort_03', 'vort_04', 'vort_05', 'wave_01', 'wave_02', 'wave_04', 'wave_05']
Val trajectories  : ['gauss_04', 'mix_04', 'mix_05', 'wave_00', 'wave_03']
Test trajectories : ['test_modon_00', 'test_modon_01', 'test_rh_00', 'test_rh_01']
[ep 000] train=2.1503e+00 val=2.4917e+00 (eta tr/va=9.962e-01/1.234e+00, uv tr/va=9.492e-01/8.418e-01, mass tr/va=2.049e+01/4.160e+01, phys tr/va=1.799e-05/4.448e-05) lr=2.00e-03
[ep 010] train=9.4725e-01 val=1.4693e+00 (eta tr/va=5.601e-01/9.361e-01, uv tr/va=3.503e-01/4.856e-01, mass tr/va=3.690e+00/4.756e+00, phys tr/va=1.620e-05/3.557e-05) lr=2.00e-03
[ep 020] train=7.5971e-01 val=1.3979e+00 (eta tr/va=4.895e-01/8.746e-01, uv tr/v

# Moving to GNN - CNN has limitations and physics did not help

# train_gnn_emulator_1layer.py

In [ ]:
# ============================================================
# train_gnn_emulator_1layer.py
# ------------------------------------------------------------
# GNN emulator for 1-layer shallow-water forecasting on
# Klein/twisted beta-plane.
#
# Input node features:
#   [eta, uc, vc, f, y_norm]
#
# Target node outputs:
#   [eta, uc, vc] at next saved time
#
# Uses 8-neighbor graph on the 2D grid.
#
# Saves:
#   best_model.pt
#   norm_stats.npz
#   loss_history.csv
#   loss_curve.png
#   metrics.txt
#
# Colab:
#   1) pip install torch-geometric if needed
#   2) !python /content/train_gnn_emulator_1layer.py
# ============================================================

import os
import csv
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_float32_matmul_precision("high")

# -------------------------
# Try PyG import
# -------------------------
try:
    from torch_geometric.data import Data
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import SAGEConv
except Exception as e:
    raise ImportError(
        "torch-geometric is required. In Colab run:\n"
        "!pip install torch-geometric\n"
        f"\nOriginal import error:\n{e}"
    )

# -------------------------
# Paths
# -------------------------
ROOT = "/content/drive/MyDrive/AI_4DVAR"
ROOT_DATA = f"{ROOT}/klein_ml_1L"
OUT_DIR   = f"{ROOT}/klein_ml_1L_ckpt_gnn"
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_PATH = os.path.join(ROOT_DATA, "dataset_train_1L.npz")
VAL_PATH   = os.path.join(ROOT_DATA, "dataset_val_1L.npz")
TEST_PATH  = os.path.join(ROOT_DATA, "dataset_test_1L.npz")

# -------------------------
# Config
# -------------------------
class CFG:
    epochs = 150
    batch_size = 2
    lr = 2e-3
    weight_decay = 1e-6
    print_every = 5

    hidden = 96
    layers = 3
    grad_clip = 1.0

    lambda_eta = 1.0
    lambda_uv  = 1.0
    lambda_mass = 1e-2

cfg = CFG()

# -------------------------
# Load datasets
# -------------------------
def load_dataset(path):
    d = np.load(path, allow_pickle=True)
    return {
        "X": d["X"].astype(np.float32),       # (N, 5, ny, nx)
        "Y": d["Y"].astype(np.float32),       # (N, 3, ny, nx)
        "ic_key": d["ic_key"],
        "step0": d["step0"].astype(np.int32),
        "step1": d["step1"].astype(np.int32),
    }

train_data = load_dataset(TRAIN_PATH)
val_data   = load_dataset(VAL_PATH)
test_data  = load_dataset(TEST_PATH)

print("Train X:", train_data["X"].shape, "Y:", train_data["Y"].shape)
print("Val   X:", val_data["X"].shape,   "Y:", val_data["Y"].shape)
print("Test  X:", test_data["X"].shape,  "Y:", test_data["Y"].shape)

Ntrain, Cin, ny, nx = train_data["X"].shape
Nval = val_data["X"].shape[0]
Ntest = test_data["X"].shape[0]
Nnodes = ny * nx

# -------------------------
# Normalization from TRAIN only
# -------------------------
def compute_channel_stats(X, Y):
    x_mean = X.mean(axis=(0,2,3), keepdims=True)
    x_std  = X.std(axis=(0,2,3), keepdims=True) + 1e-6
    y_mean = Y.mean(axis=(0,2,3), keepdims=True)
    y_std  = Y.std(axis=(0,2,3), keepdims=True) + 1e-6
    return x_mean.astype(np.float32), x_std.astype(np.float32), y_mean.astype(np.float32), y_std.astype(np.float32)

x_mean, x_std, y_mean, y_std = compute_channel_stats(train_data["X"], train_data["Y"])

np.savez_compressed(
    os.path.join(OUT_DIR, "norm_stats.npz"),
    x_mean=x_mean, x_std=x_std,
    y_mean=y_mean, y_std=y_std,
)
print("Saved normalization stats.")

def normalize_X(X):
    return (X - x_mean) / x_std

def normalize_Y(Y):
    return (Y - y_mean) / y_std

def denormalize_Y(Yn):
    return Yn * y_std + y_mean

train_Xn = normalize_X(train_data["X"])
train_Yn = normalize_Y(train_data["Y"])
val_Xn   = normalize_X(val_data["X"])
val_Yn   = normalize_Y(val_data["Y"])
test_Xn  = normalize_X(test_data["X"])
test_Yn  = normalize_Y(test_data["Y"])

# for mass penalty
y_mean_t = torch.tensor(y_mean, device=device)
y_std_t  = torch.tensor(y_std, device=device)

# -------------------------
# Build 8-neighbor edge index
# -------------------------
def node_id(i, j):
    return i * nx + j

def build_edge_index(ny, nx):
    edges = []
    for i in range(ny):
        for j in range(nx):
            src = node_id(i, j)
            for di in (-1, 0, 1):
                for dj in (-1, 0, 1):
                    if di == 0 and dj == 0:
                        continue
                    ii = i + di
                    jj = (j + dj) % nx   # periodic in x
                    if ii < 0 or ii >= ny:
                        continue         # nonperiodic in y
                    dst = node_id(ii, jj)
                    edges.append([src, dst])
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    return edge_index

EDGE_INDEX = build_edge_index(ny, nx)
print("Edge index:", EDGE_INDEX.shape)

# -------------------------
# PyG dataset
# -------------------------
def sample_to_graph(Xs, Ys, edge_index, ic_key=None, step0=None, step1=None):
    """
    Xs: (5,ny,nx), Ys: (3,ny,nx)
    """
    x = torch.from_numpy(Xs.reshape(5, -1).T).float()   # (Nnodes, 5)
    y = torch.from_numpy(Ys.reshape(3, -1).T).float()   # (Nnodes, 3)

    data = Data(x=x, y=y, edge_index=edge_index)
    data.ny = ny
    data.nx = nx
    data.ic_key = str(ic_key) if ic_key is not None else ""
    data.step0 = int(step0) if step0 is not None else -1
    data.step1 = int(step1) if step1 is not None else -1
    return data

def build_graph_dataset(data_dict, Xn, Yn, edge_index):
    ds = []
    for i in range(len(Xn)):
        ds.append(
            sample_to_graph(
                Xn[i], Yn[i], edge_index,
                ic_key=data_dict["ic_key"][i],
                step0=data_dict["step0"][i],
                step1=data_dict["step1"][i],
            )
        )
    return ds

train_ds = build_graph_dataset(train_data, train_Xn, train_Yn, EDGE_INDEX)
val_ds   = build_graph_dataset(val_data,   val_Xn,   val_Yn,   EDGE_INDEX)
test_ds  = build_graph_dataset(test_data,  test_Xn,  test_Yn,  EDGE_INDEX)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False)

# -------------------------
# Model
# -------------------------
class GNNEmulator(nn.Module):
    def __init__(self, in_ch=5, hidden=96, out_ch=3):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.conv3 = SAGEConv(hidden, hidden)
        self.head  = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, out_ch),
        )

    def forward(self, x, edge_index):
        z = self.conv1(x, edge_index)
        z = torch.gelu(z)
        z = self.conv2(z, edge_index)
        z = torch.gelu(z)
        z = self.conv3(z, edge_index)
        z = torch.gelu(z)
        return self.head(z)

model = GNNEmulator(in_ch=5, hidden=cfg.hidden, out_ch=3).to(device)

# -------------------------
# Loss
# -------------------------
def loss_fn(pred_n, targ_n, batch_vec):
    """
    pred_n, targ_n: (total_nodes_in_batch, 3) normalized
    batch_vec:      (total_nodes_in_batch,) graph membership
    """
    diff = pred_n - targ_n
    loss_eta = torch.mean(diff[:, 0:1]**2)
    loss_uv  = torch.mean(diff[:, 1:3]**2)

    # mass penalty on denormalized eta, per graph
    pred_eta = pred_n[:, 0:1] * y_std_t[:, 0:1].reshape(1,1) + y_mean_t[:, 0:1].reshape(1,1)
    targ_eta = targ_n[:, 0:1] * y_std_t[:, 0:1].reshape(1,1) + y_mean_t[:, 0:1].reshape(1,1)

    ng = int(batch_vec.max().item()) + 1
    mass_pen = 0.0
    for g in range(ng):
        mask = (batch_vec == g)
        pe = pred_eta[mask].mean()
        te = targ_eta[mask].mean()
        mass_pen = mass_pen + (pe - te)**2
    mass_pen = mass_pen / max(ng, 1)

    loss = cfg.lambda_eta * loss_eta + cfg.lambda_uv * loss_uv + cfg.lambda_mass * mass_pen
    comps = {
        "eta": loss_eta.detach().item(),
        "uv": loss_uv.detach().item(),
        "mass": mass_pen.detach().item(),
    }
    return loss, comps

# -------------------------
# Optimizer
# -------------------------
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=12
)

# -------------------------
# Epoch loops
# -------------------------
def run_epoch(loader, training):
    if training:
        model.train()
    else:
        model.eval()

    loss_sum = 0.0
    eta_sum = 0.0
    uv_sum = 0.0
    mass_sum = 0.0
    n = 0

    for batch in loader:
        batch = batch.to(device)

        with torch.set_grad_enabled(training):
            pred = model(batch.x, batch.edge_index)
            loss, comps = loss_fn(pred, batch.y, batch.batch)

            if training:
                if torch.isfinite(loss):
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                    optimizer.step()

        bs = batch.num_graphs
        loss_sum += loss.detach().item() * bs
        eta_sum  += comps["eta"] * bs
        uv_sum   += comps["uv"] * bs
        mass_sum += comps["mass"] * bs
        n += bs

    return {
        "loss": loss_sum / max(n, 1),
        "eta": eta_sum / max(n, 1),
        "uv": uv_sum / max(n, 1),
        "mass": mass_sum / max(n, 1),
    }

# -------------------------
# Train loop
# -------------------------
history = []
best_val = float("inf")
best_path = os.path.join(OUT_DIR, "best_model.pt")

for ep in range(cfg.epochs):
    tr = run_epoch(train_loader, training=True)
    va = run_epoch(val_loader, training=False)

    scheduler.step(va["loss"])

    history.append({
        "epoch": ep,
        "train": tr["loss"],
        "val": va["loss"],
        "train_eta": tr["eta"],
        "train_uv": tr["uv"],
        "train_mass": tr["mass"],
        "val_eta": va["eta"],
        "val_uv": va["uv"],
        "val_mass": va["mass"],
        "lr": optimizer.param_groups[0]["lr"],
    })

    if va["loss"] < best_val:
        best_val = va["loss"]
        torch.save(model.state_dict(), best_path)

    if (ep % cfg.print_every) == 0 or ep == cfg.epochs - 1:
        print(
            f"[ep {ep:03d}] "
            f"train={tr['loss']:.4e} val={va['loss']:.4e} "
            f"(eta tr/va={tr['eta']:.3e}/{va['eta']:.3e}, "
            f"uv tr/va={tr['uv']:.3e}/{va['uv']:.3e}, "
            f"mass tr/va={tr['mass']:.3e}/{va['mass']:.3e}) "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

print("Best val loss:", best_val)
print("Saved:", best_path)

# -------------------------
# Save history
# -------------------------
hist_csv = os.path.join(OUT_DIR, "loss_history.csv")
with open(hist_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    w.writeheader()
    w.writerows(history)
print("Saved:", hist_csv)

plt.figure(figsize=(7,4))
plt.plot([h["epoch"] for h in history], [h["train"] for h in history], label="train")
plt.plot([h["epoch"] for h in history], [h["val"] for h in history], label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("GNN emulator loss")
plt.tight_layout()
loss_png = os.path.join(OUT_DIR, "loss_curve.png")
plt.savefig(loss_png, dpi=140)
plt.close()
print("Saved:", loss_png)

# -------------------------
# Reload best model
# -------------------------
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

# -------------------------
# Predict whole dataset
# -------------------------
def predict_graph_dataset(ds):
    preds = []
    with torch.no_grad():
        for data in ds:
            data = data.to(device)
            yp = model(data.x, data.edge_index).cpu().numpy()      # (Nnodes, 3)
            yp = yp.T.reshape(3, ny, nx)
            preds.append(yp)
    preds = np.stack(preds, axis=0)                                # (N, 3, ny, nx)
    return preds

val_pred_n  = predict_graph_dataset(val_ds)
test_pred_n = predict_graph_dataset(test_ds)

val_pred  = denormalize_Y(val_pred_n)
test_pred = denormalize_Y(test_pred_n)

val_true  = val_data["Y"]
test_true = test_data["Y"]

# -------------------------
# Metrics
# -------------------------
def channel_rmse(pred, true):
    out = {}
    names = ["eta", "uc", "vc"]
    for i, nm in enumerate(names):
        out[nm] = float(np.sqrt(np.mean((pred[:, i] - true[:, i])**2)))
    return out

val_rmse = channel_rmse(val_pred, val_true)
test_rmse = channel_rmse(test_pred, test_true)

print("\nValidation RMSE:", val_rmse)
print("Test RMSE:", test_rmse)

with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write("Validation RMSE\n")
    for k, v in val_rmse.items():
        f.write(f"{k}: {v:.6e}\n")
    f.write("\nTest RMSE\n")
    for k, v in test_rmse.items():
        f.write(f"{k}: {v:.6e}\n")
print("Saved metrics.txt")

# -------------------------
# Prediction plots
# -------------------------
pred_dir = os.path.join(OUT_DIR, "predictions")
os.makedirs(pred_dir, exist_ok=True)

def plot_sample(idx, pred, true, split_name, meta):
    names = ["eta", "uc", "vc"]
    fig, axs = plt.subplots(3, 3, figsize=(11, 10))

    for i, nm in enumerate(names):
        im = axs[i,0].imshow(true[i], origin="lower")
        axs[i,0].set_title(f"Truth {nm}")
        plt.colorbar(im, ax=axs[i,0], fraction=0.046, pad=0.04)

        im = axs[i,1].imshow(pred[i], origin="lower")
        axs[i,1].set_title(f"Pred {nm}")
        plt.colorbar(im, ax=axs[i,1], fraction=0.046, pad=0.04)

        err = pred[i] - true[i]
        im = axs[i,2].imshow(err, origin="lower")
        axs[i,2].set_title(f"Error {nm}")
        plt.colorbar(im, ax=axs[i,2], fraction=0.046, pad=0.04)

    ic = meta["ic_key"][idx]
    s0 = int(meta["step0"][idx])
    s1 = int(meta["step1"][idx])
    plt.suptitle(f"{split_name}: {ic}  {s0}->{s1}", y=0.995)
    plt.tight_layout()
    out = os.path.join(pred_dir, f"{split_name}_{idx:03d}_{ic}_{s0}_{s1}.png")
    plt.savefig(out, dpi=140)
    plt.close()

for i in range(min(4, len(val_pred))):
    plot_sample(i, val_pred[i], val_true[i], "val", val_data)

for i in range(min(6, len(test_pred))):
    plot_sample(i, test_pred[i], test_true[i], "test", test_data)

print("Saved prediction plots in:", pred_dir)

# -------------------------
# Rollout-like evaluation
# -------------------------
def group_by_ic(meta):
    out = {}
    for i, ic in enumerate(meta["ic_key"]):
        out.setdefault(str(ic), []).append(i)
    return out

def make_graph_from_raw_X(Xraw):
    x = Xraw.reshape(5, -1).T.astype(np.float32)
    data = Data(
        x=torch.from_numpy(x),
        edge_index=EDGE_INDEX
    )
    return data

def rollout_like(meta, X_raw, Y_true, split_name):
    groups = group_by_ic(meta)
    summary_lines = []

    roll_dir = os.path.join(OUT_DIR, "rollout_like")
    os.makedirs(roll_dir, exist_ok=True)

    for ic in sorted(groups.keys()):
        idxs = sorted(groups[ic], key=lambda j: meta["step0"][j])

        x_cur_raw = X_raw[idxs[0]:idxs[0]+1].copy()
        pred_traj = []
        true_traj = []

        with torch.no_grad():
            for p, j in enumerate(idxs):
                x_cur_n = normalize_X(x_cur_raw)[0]
                g = make_graph_from_raw_X(x_cur_n).to(device)

                yp_n = model(g.x, g.edge_index).cpu().numpy()      # (Nnodes,3)
                yp_n = yp_n.T.reshape(3, ny, nx)                   # (3,ny,nx)
                yp = denormalize_Y(yp_n[None])[0]

                pred_traj.append(yp)
                true_traj.append(Y_true[j])

                if p != len(idxs) - 1:
                    x_next_raw = X_raw[j+1:j+2].copy()
                    x_next_raw[0, 0:3] = yp
                    x_cur_raw = x_next_raw

        pred_traj = np.stack(pred_traj, axis=0)
        true_traj = np.stack(true_traj, axis=0)

        eta_rmse = [math.sqrt(np.mean((pred_traj[k,0] - true_traj[k,0])**2)) for k in range(len(pred_traj))]
        uc_rmse  = [math.sqrt(np.mean((pred_traj[k,1] - true_traj[k,1])**2)) for k in range(len(pred_traj))]
        vc_rmse  = [math.sqrt(np.mean((pred_traj[k,2] - true_traj[k,2])**2)) for k in range(len(pred_traj))]

        plt.figure(figsize=(7,4))
        plt.plot(range(1, len(eta_rmse)+1), eta_rmse, "-o", label="eta")
        plt.plot(range(1, len(uc_rmse)+1), uc_rmse, "-s", label="uc")
        plt.plot(range(1, len(vc_rmse)+1), vc_rmse, "-^", label="vc")
        plt.xlabel("macro lead step")
        plt.ylabel("RMSE")
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.title(f"{split_name} rollout-like RMSE: {ic}")
        plt.tight_layout()
        plt.savefig(os.path.join(roll_dir, f"{split_name}_{ic}_rmse.png"), dpi=140)
        plt.close()

        summary_lines.append(
            f"{split_name:5s} {ic:16s}  "
            f"final_eta_rmse={eta_rmse[-1]:.6e}  "
            f"final_uc_rmse={uc_rmse[-1]:.6e}  "
            f"final_vc_rmse={vc_rmse[-1]:.6e}"
        )

    return summary_lines

val_rollout_summary = rollout_like(val_data, val_data["X"], val_true, "val")
test_rollout_summary = rollout_like(test_data, test_data["X"], test_true, "test")

with open(os.path.join(OUT_DIR, "rollout_like_summary.txt"), "w") as f:
    f.write("\n".join(val_rollout_summary + [""] + test_rollout_summary))

print("Saved rollout_like_summary.txt")
print("Done.")